# RetailHero Purchase Uplift Modeling Experiment

## Problem Definition

The RetailHero dataset was released by X5 Retail Group for an uplift-modeling competition on promotional communication.

The business setting is customer targeting for an advertising SMS campaign.

A conventional response model asks:

> Which customers are most likely to make a purchase?

For campaign targeting, however, the more relevant question is:

> Which customers are more likely to make a purchase **because they receive the promotional communication**?

A customer who is already very likely to purchase may receive a high response score even if the campaign does not change their behavior. Uplift modeling instead focuses on the **incremental effect of treatment**.

The objective of this experiment is therefore to compare conventional response-based targeting with uplift-based targeting.

The main decision question is:

> **Under a limited targeting budget, which policy selects customers with the highest incremental purchase outcome?**

The experiment compares:

* **Response Model** as the conventional targeting baseline;
* **T-Learner** as an uplift-model candidate;
* **X-Learner** as an uplift-model candidate.

The goal is not exhaustive hyperparameter optimization. The goal is to compare these modeling approaches under the same prepared data structure, split strategy, validation protocol, and targeting budgets.


## RetailHero Data Structure

Unlike Hillstrom, RetailHero is not provided as a single customer-level modeling table.

The raw data is distributed across four tables:

* `clients`: customer information;
* `products`: product metadata;
* `purchases`: historical customer transactions;
* `uplift_train`: customers in the labeled uplift experiment, including treatment assignment and observed outcome.

The tables play different roles but are connected through customer and product identifiers.

At a high level:

  ```text
  clients
    │
    │ client_id
    ▼
  purchases ◄──── product_id ──── products
    │
    │ client_id
    ▼
  uplift_train
  ```

The historical customer, purchase, and product information will be used to construct **pre-treatment customer features**.

The `uplift_train` table provides the two variables required for uplift modeling:

  ```text
  treatment = treatment_flg
  outcome   = target
  ```

The final modeling dataset will therefore contain **one row per customer**, with:


> customer features + treatment + outcome


The raw RetailHero tables cannot be passed directly to the reusable uplift framework because they must first be understood, validated, and transformed into this customer-level structure.

The original competition `uplift_test` and submission files are not required for this experiment. The labeled `uplift_train` population is used to construct the modeling dataset and an internal train / validation / locked-test split.


In [2]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

COLORS = {
    "pacific_green": "#017360",
    "rainy_lake": "#446E8F",
    "serene_sea": "#77A7C4",
    "sora_blue": "#A1DBF1",
    "afterglow": "#F2E6CE",
    "bungalow_maple": "#F3D094",
}

PALETTE = list(COLORS.values())

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize":11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [3]:
TABLE_HIGHLIGHT = {
    "blue": "background-color: #1976D2; color: white; font-weight: 700;",
    "red": "background-color: #D32F2F; color: white; font-weight: 700;",
}


def style_table(df, formats=None):
    return (
        df.style
        .hide(axis="index")
        .format(
            formats or {},
            na_rep="—",
        )
    )


def style_properties(style_key):
    return {
        item.split(":")[0].strip(): item.split(":")[1].strip()
        for item in TABLE_HIGHLIGHT[style_key].rstrip(";").split(";")
    }


def paint(styler, rows, columns, style_key):
    rows = list(rows)
    columns = [columns] if isinstance(columns, str) else list(columns)

    if not rows:
        return styler

    return styler.set_properties(
        subset=pd.IndexSlice[rows, columns],
        **style_properties(style_key),
    )


In [4]:
PROJECT_ROOT = Path.cwd().parents[1]

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "retailhero"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim" / "retailhero"

RAW_FILES = {
    "clients": RAW_DATA_DIR / "clients.csv",
    "products": RAW_DATA_DIR / "products.csv",
    "purchases": RAW_DATA_DIR / "purchases.csv",
    "uplift_train": RAW_DATA_DIR / "uplift_train.csv",
}

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, path in RAW_FILES.items():
    assert path.exists(), f"Missing {name}: {path}"

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Interim data:", INTERIM_DATA_DIR)

Project root: d:\thao\d\uplif_model\uplif_customer_selection
Raw data: d:\thao\d\uplif_model\uplif_customer_selection\data\raw\retailhero
Interim data: d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero


In [5]:
file_inventory = pd.DataFrame([
    {
        "table": table_name,
        "file": path.name,
        "size_mb": path.stat().st_size / (1024**2),
        "path": str(path),
    }
    for table_name, path in RAW_FILES.items()
]).sort_values("size_mb", ascending=False)

display(file_inventory)

,table,file,size_mb,path
2,purchases,purchases.csv,"4,256.9881",d:\thao\d\uplif_model\uplif_customer_selection...
0,clients,clients.csv,20.7293,d:\thao\d\uplif_model\uplif_customer_selection...
1,products,products.csv,3.7101,d:\thao\d\uplif_model\uplif_customer_selection...
3,uplift_train,uplift_train.csv,2.8616,d:\thao\d\uplif_model\uplif_customer_selection...


## DuckDB Loading Strategy

The raw RetailHero files are intentionally **not loaded into pandas as full tables.**

This is particularly important for `purchases`, which is much larger than the other tables.

The notebook uses the following loading strategy:

1. Keep the original CSV files unchanged under `data/raw/retailhero/`.
2. Use DuckDB to query the raw tables.
3. Convert the large `purchases.csv` file once into a local Parquet cache.
4. Query subsequent `purchases` data from the Parquet version through DuckDB.
5. Avoid loading the complete `purchases` table into pandas.
6. Convert only small query results, summaries, samples, or final customer-level feature tables to pandas.

The Parquet file is used only as an analytical cache. The original CSV remains the raw source of truth.

This approach avoids repeatedly parsing the large CSV while allowing DuckDB to read only the required columns and rows.

The remaining RetailHero tables are much smaller and can remain as DuckDB views over their original CSV files.


In [6]:
DUCKDB_CACHE_DIR = INTERIM_DATA_DIR / "_duckdb_cache"
DUCKDB_TEMP_DIR = DUCKDB_CACHE_DIR / "tmp"

DUCKDB_CACHE_DIR.mkdir(parents=True,exist_ok=True,)
DUCKDB_TEMP_DIR.mkdir(parents=True,exist_ok=True,)

duckdb_connection = duckdb.connect(database=":memory:")


def to_sql_path(path: Path) -> str:
    """Return an absolute path escaped for use inside a SQL string literal."""
    return (
        path.resolve()
        .as_posix()
        .replace("'", "''")
    )


duckdb_connection.execute(
    f"SET temp_directory = '{to_sql_path(DUCKDB_TEMP_DIR)}'"
)

print("DuckDB temp directory:",DUCKDB_TEMP_DIR,)


DuckDB temp directory: d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\_duckdb_cache\tmp


In [7]:
PURCHASES_PARQUET_PATH = (DUCKDB_CACHE_DIR / "purchases.parquet")

if not PURCHASES_PARQUET_PATH.exists():
    print("Creating purchases Parquet cache...")

    duckdb_connection.execute(
        f"""
        COPY (
            SELECT *
            FROM read_csv(
                '{to_sql_path(RAW_FILES["purchases"])}',
                header = true
            )
        )
        TO '{to_sql_path(PURCHASES_PARQUET_PATH)}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )
else:
    print("Using existing purchases Parquet cache.")

Using existing purchases Parquet cache.


In [8]:
CSV_VIEW_TABLES = [
    "clients",
    "products",
    "uplift_train",
]

for table_name in CSV_VIEW_TABLES:
    file_path = RAW_FILES[table_name]

    duckdb_connection.execute(
        f"""
        CREATE OR REPLACE VIEW {table_name}_raw AS
        SELECT *
        FROM read_csv(
            '{to_sql_path(file_path)}',
            header = true,
            sample_size = 100000,
            strict_mode = true
        )
        """
    )

# The large purchases table is queried from the Parquet cache.
duckdb_connection.execute(
    f"""
    CREATE OR REPLACE VIEW purchases_raw AS
    SELECT *
    FROM read_parquet(
        '{to_sql_path(PURCHASES_PARQUET_PATH)}'
    )
    """
)

registered_relations = duckdb_connection.execute(
    "SHOW TABLES"
).df()

display(registered_relations)

,name
0,clients_raw
1,products_raw
2,purchases_raw
3,uplift_train_raw




## Data Overview

### Table and Column Description

The RetailHero dataset contains customer information, product metadata, historical purchase transactions, and uplift-experiment labels.

The competition documentation describes the general role of each table but does not provide a detailed business definition for every raw column.

The descriptions below therefore represent the **initial working interpretation** based on the available documentation, column names, and apparent table structure.

Before cleaning, EDA, or feature engineering, the notebook will check whether the observed data is consistent with these assumptions.

### `clients`

One row is expected to represent one customer.

| Column              | Description                                                        |
| ------------------- | ------------------------------------------------------------------ |
| `client_id`         | Unique customer identifier used to link tables                     |
| `first_issue_date`  | Date associated with the customer's first loyalty-card issuance    |
| `first_redeem_date` | Date associated with the customer's first loyalty-point redemption |
| `age`               | Customer age                                                       |
| `gender`            | Customer gender                                                    |

### `products`

One row is expected to represent one product.

| Column             | Description                                             |
| ------------------ | ------------------------------------------------------- |
| `product_id`       | Unique product identifier used to link purchase records |
| `level_1`          | Highest-level product category                          |
| `level_2`          | Second-level product category                           |
| `level_3`          | Third-level product category                            |
| `level_4`          | Most detailed product category provided                 |
| `segment_id`       | Product segment identifier                              |
| `brand_id`         | Brand identifier                                        |
| `vendor_id`        | Vendor or supplier identifier                           |
| `netto`            | Product net-weight value                                |
| `is_own_trademark` | Whether the product belongs to the retailer's own brand |
| `is_alcohol`       | Whether the product is alcoholic                        |

### `purchases`

The initial interpretation is that one row represents one **product line within a transaction**.

A single transaction may therefore appear across multiple rows when several products are purchased.

| Column                    | Description                                                          |
| ------------------------- | -------------------------------------------------------------------- |
| `client_id`               | Customer associated with the transaction                             |
| `transaction_id`          | Transaction or receipt identifier                                    |
| `transaction_datetime`    | Date and time of the transaction                                     |
| `regular_points_received` | Regular loyalty points received                                      |
| `express_points_received` | Express loyalty points received                                      |
| `regular_points_spent`    | Regular loyalty points redeemed                                      |
| `express_points_spent`    | Express loyalty points redeemed                                      |
| `purchase_sum`            | Purchase amount recorded for the transaction                         |
| `store_id`                | Store identifier                                                     |
| `product_id`              | Purchased product identifier                                         |
| `product_quantity`        | Quantity of the product purchased                                    |
| `trn_sum_from_iss`        | Amount associated with the product line on the point-issuance side   |
| `trn_sum_from_red`        | Amount associated with the product line on the point-redemption side |

The exact business rules behind `express_points_*`, `trn_sum_from_iss`, and `trn_sum_from_red` are not documented in detail in the available competition description.

The notebook will therefore inspect the observed transaction structure before assigning these fields a more specific interpretation or using them for feature engineering.

### `uplift_train`

One row is expected to represent one customer in the labeled uplift experiment.

| Column          | Description                                                                    |
| --------------- | ------------------------------------------------------------------------------ |
| `client_id`     | Customer identifier used to join with customer and purchase data               |
| `treatment_flg` | Treatment assignment: `1` if promotional communication was sent, otherwise `0` |
| `target`        | Binary outcome indicating whether the customer made a purchase afterward       |

For this experiment:

```text
treatment = treatment_flg
outcome   = target
```

The remaining tables provide information that can potentially be transformed into **pre-treatment customer features**, while `uplift_train` determines the treatment and observed outcome for each modeled customer.


In [9]:
RELATIONS = {
    "clients": "clients_raw",
    "products": "products_raw",
    "purchases": "purchases_raw",
    "uplift_train": "uplift_train_raw",
}

table_shapes = []

for table_name, relation_name in RELATIONS.items():
    row_count = duckdb_connection.execute(
        f"SELECT COUNT(*) FROM {relation_name}"
    ).fetchone()[0]

    schema = duckdb_connection.execute(
        f"DESCRIBE {relation_name}"
    ).df()

    table_shapes.append({
        "table": table_name,
        "rows": row_count,
        "columns": len(schema),
    })

table_shapes = pd.DataFrame(table_shapes)

display(table_shapes)

,table,rows,columns
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_train,200039,3


In [10]:
for table_name, relation_name in RELATIONS.items():
    print(f"\n{table_name}")
    display(
        duckdb_connection.execute(
            f"DESCRIBE {relation_name}"
        ).df()
    )


clients


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,first_issue_date,TIMESTAMP,YES,None,None,None
2,first_redeem_date,TIMESTAMP,YES,None,None,None
3,age,BIGINT,YES,None,None,None
4,gender,VARCHAR,YES,None,None,None



products


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,level_1,VARCHAR,YES,None,None,None
2,level_2,VARCHAR,YES,None,None,None
3,level_3,VARCHAR,YES,None,None,None
4,level_4,VARCHAR,YES,None,None,None
5,segment_id,DOUBLE,YES,None,None,None
6,brand_id,VARCHAR,YES,None,None,None
7,vendor_id,VARCHAR,YES,None,None,None
8,netto,DOUBLE,YES,None,None,None
9,is_own_trademark,BIGINT,YES,None,None,None



purchases


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,transaction_id,VARCHAR,YES,None,None,None
2,transaction_datetime,TIMESTAMP,YES,None,None,None
3,regular_points_received,DOUBLE,YES,None,None,None
4,express_points_received,DOUBLE,YES,None,None,None
5,regular_points_spent,DOUBLE,YES,None,None,None
6,express_points_spent,DOUBLE,YES,None,None,None
7,purchase_sum,DOUBLE,YES,None,None,None
8,store_id,VARCHAR,YES,None,None,None
9,product_id,VARCHAR,YES,None,None,None



uplift_train


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,treatment_flg,BIGINT,YES,None,None,None
2,target,BIGINT,YES,None,None,None


In [11]:
for table_name in (
    "clients",
    "products",
    "purchases",
    "uplift_train",
):
    print(f"\n{table_name}")

    display(
        duckdb_connection.execute(
            f"SELECT * FROM {RELATIONS[table_name]} LIMIT 5"
        ).df()
    )


clients


,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaT,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U



products


,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto,is_own_trademark,is_alcohol
0,0003020d3c,c3d3a8e8c6,c2a3ea8d5e,b7cda0ec0c,6376f2a852,123.0000,394a54a7c1,9eaff48661,0.4000,0,0
1,0003870676,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,acd3dd483f,10486c3cf0,0.6800,0,0
2,0003ceaf69,c3d3a8e8c6,f2333c90fb,419bc5b424,f6148afbc0,271.0000,f597581079,764e660dda,0.5000,0,0
3,000701e093,ec62ce61e3,4202626fcb,88a515c084,48cf3d488f,172.0000,54a90fe769,03c2d70bad,0.1120,0,0
4,0007149564,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,63417fe1f3,f329130198,0.6000,0,0



purchases


,client_id,transaction_id,transaction_datetime,regular_points_received,express_points_received,regular_points_spent,express_points_spent,purchase_sum,store_id,product_id,product_quantity,trn_sum_from_iss,trn_sum_from_red
0,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,9a80204f78,2.0000,80.0000,NaN
1,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,da89ebd374,1.0000,65.0000,NaN
2,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,0a95e1151d,1.0000,24.0000,NaN
3,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,4055b15e4a,2.0000,50.0000,NaN
4,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,a685f1916b,1.0000,22.0000,NaN



uplift_train


,client_id,treatment_flg,target
0,000012768d,0,1
1,000036f903,1,1
2,00010925a5,1,1
3,0001f552b0,1,1
4,00020e7b18,1,1


## Structural Data Quality Checks

Before EDA or feature engineering, validate both dataset structure and value quality:

1. Primary keys are unique and non-null where required;
2. Treatment and target contain valid binary values;
3. Relationships between tables do not contain orphan records;
4. Numeric columns have plausible distributions and ranges;
5. Missing values are understood for every column;
6. Categorical values are internally consistent;
7. Explicit domain constraints are satisfied;
8. Datetime values and date relationships are plausible.


In [12]:
validation_issues = []

UNIQUE_KEYS = {
    "clients": "client_id",
    "products": "product_id",
    "uplift_train": "client_id",
}

key_profiles = []

for table_name, key in UNIQUE_KEYS.items():
    rows, null_keys, distinct_keys = duckdb_connection.execute(
        f""" 
        SELECT
            COUNT(*),
            COUNT(*) FILTER (WHERE {key} IS NULL),
            COUNT(DISTINCT {key})
        FROM {RELATIONS[table_name]}
        """
    ).fetchone()

    duplicate_keys = rows - null_keys - distinct_keys

    key_profiles.append({
        "table": table_name,
        "rows": rows,
        "distinct_keys": distinct_keys,
        "null_keys": null_keys,
        "duplicate_keys": duplicate_keys,
    })

    if null_keys or duplicate_keys:
        validation_issues.append(
            f"{table_name}.{key}: {null_keys} null keys, {duplicate_keys} duplicate keys"
    )

display(pd.DataFrame(key_profiles))

,table,rows,distinct_keys,null_keys,duplicate_keys
0,clients,400162,400162,0,0
1,products,43038,43038,0,0
2,uplift_train,200039,200039,0,0


In [13]:
FULL_ROW_DUPLICATE_TABLES = [
    "clients",
    "products",
    "uplift_train",
]

full_row_duplicate_profiles = []

for table_name in FULL_ROW_DUPLICATE_TABLES:
    relation_name = RELATIONS[table_name]

    row_counts = duckdb_connection.execute(
        f"""
        WITH distinct_rows AS (
            SELECT DISTINCT *
            FROM {relation_name}
        )

        SELECT
            (SELECT COUNT(*) FROM {relation_name}) AS rows,
            (SELECT COUNT(*) FROM distinct_rows) AS distinct_rows
        """
    ).fetchone()

    rows, distinct_rows = row_counts
    duplicate_rows = rows - distinct_rows

    full_row_duplicate_profiles.append({
        "table": table_name,
        "rows": rows,
        "distinct_rows": distinct_rows,
        "duplicate_rows": duplicate_rows,
    })

    if duplicate_rows:
        validation_issues.append(
            f"{table_name}: {duplicate_rows} duplicate full rows"
        )

full_row_duplicate_profile = pd.DataFrame(full_row_duplicate_profiles)

display(full_row_duplicate_profile)


,table,rows,distinct_rows,duplicate_rows
0,clients,400162,400162,0
1,products,43038,43038,0
2,uplift_train,200039,200039,0


In [14]:
treatment_target_check = duckdb_connection.execute(
    """
    SELECT 
        COUNT(*) FILTER (WHERE treatment_flg IS NULL) AS treatment_nulls,
        COUNT(*) FILTER(
            WHERE treatment_flg IS NOT NULL
                AND treatment_flg NOT IN (0,1)
        ) AS treatment_invalid,
        COUNT(*) FILTER (WHERE target IS NULL) AS target_nulls,
        COUNT(*) FILTER (
            WHERE target IS NOT NULL
                AND target NOT IN (0,1)
        ) AS target_invalid
    FROM uplift_train_raw
    """
).df()

display(treatment_target_check)

if treatment_target_check.iloc[0].sum() > 0:
    validation_issues.append("uplift_train: invalid or missing treatment/target values")

treatment_target_profile = duckdb_connection.execute(
    """
    SELECT
        treatment_flg,
        target,
        COUNT(*) AS customers,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS proportion
    FROM uplift_train_raw
    GROUP BY treatment_flg, target
    ORDER BY treatment_flg, target
    """
).df()

display(treatment_target_profile)


,treatment_nulls,treatment_invalid,target_nulls,target_invalid
0,0,0,0,0


,treatment_flg,target,customers,proportion
0,0,0,39695,0.1984
1,0,1,60363,0.3018
2,1,0,36342,0.1817
3,1,1,63639,0.3181


In [15]:
uplift_relationship_check = duckdb_connection.execute(
    """ 
    SELECT
        COUNT(*) FILTER(
            WHERE c.client_id IS NULL
        ) AS uplift_train_clients_missing_from_clients
    FROM uplift_train_raw AS u
    LEFT JOIN clients_raw AS c
        USING (client_id)
    """
).df()

purchase_relationship_check = duckdb_connection.execute(
    """ 
    SELECT
        COUNT(*) FILTER(
            WHERE c.client_id IS NULL    
        ) AS purchases_row_with_unknow_client,
        COUNT(*) FILTER(
            WHERE pr.product_id IS NULL
        ) AS purchases_rows_with_unknown_product
    FROM purchases_raw as p
    LEFT JOIN clients_raw as c
        USING (client_id)
    LEFT JOIN products_raw AS pr
        USING (product_id)
    """
).df()

display(uplift_relationship_check)
display(purchase_relationship_check)

if uplift_relationship_check.iloc[0, 0] > 0:
    validation_issues.append("uplift_train contains unknown client_id values")

if purchase_relationship_check.iloc[0].sum() > 0:
    validation_issues.append("purchases contains unknown client_id or product_id values")

,uplift_train_clients_missing_from_clients
0,0


,purchases_row_with_unknow_client,purchases_rows_with_unknown_product
0,0,0


In [16]:
purchase_history_profile = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,
        COUNT(DISTINCT client_id) AS purchasing_clients,
        COUNT(DISTINCT transaction_id) AS transactions,
        COUNT(DISTINCT product_id) AS purchased_products
    FROM purchases_raw
    """
).df()

display(purchase_history_profile)

,purchase_rows,purchasing_clients,transactions,purchased_products
0,45786568,400162,8045201,42530


In [17]:
NUMERIC_COLUMNS = {
    "clients": ["age"],
    "products": ["netto"],
    "purchases": [
        "regular_points_received",
        "express_points_received",
        "regular_points_spent",
        "express_points_spent",
        "purchase_sum",
        "product_quantity",
        "trn_sum_from_iss",
        "trn_sum_from_red",
    ],
}

CATEGORICAL_COLUMNS = {
    "clients": ["gender"],
    "products": [
        "level_1",
        "level_2",
        "level_3",
        "level_4",
        "segment_id",
        "brand_id",
        "vendor_id",
    ],
    "purchases": ["store_id"],
}

DATETIME_COLUMNS = {
    "clients": ["first_issue_date", "first_redeem_date"],
    "purchases": ["transaction_datetime"],
}

REQUIRED_NON_NULL = {
    "clients": {"client_id"},
    "products": {"product_id"},
    "purchases": {"client_id", "transaction_id", "transaction_datetime", "product_id"},
    "uplift_train": {"client_id", "treatment_flg", "target"},
}

In [18]:
NUMERIC_AGGREGATES = {
    "count": "COUNT({column})",
    "mean": "AVG({column})",
    "std": "STDDEV_SAMP({column})",
    "min": "MIN({column})",
    "25%": "APPROX_QUANTILE({column}, 0.25)",
    "50%": "APPROX_QUANTILE({column}, 0.50)",
    "75%": "APPROX_QUANTILE({column}, 0.75)",
    "max": "MAX({column})",
}


def numeric_profile(table_name, columns):
    expressions = []

    for column in columns:
        quoted = f'"{column}"'
        for metric, expression in NUMERIC_AGGREGATES.items():
            expressions.append(
                f'{expression.format(column=quoted)} AS "{column}__{metric}"'
            )

    result = duckdb_connection.execute(
        f"""
        SELECT {", ".join(expressions)}
        FROM {RELATIONS[table_name]}
        """
    ).df().iloc[0]

    return pd.DataFrame([
        {
            "table": table_name,
            "column": column,
            **{
                metric: result[f"{column}__{metric}"]
                for metric in NUMERIC_AGGREGATES
            },
        }
        for column in columns
    ])


numeric_profiles = pd.concat(
    [
        numeric_profile(table_name, columns)
        for table_name, columns in NUMERIC_COLUMNS.items()
    ],
    ignore_index=True,
)

display(numeric_profiles)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,column,count,mean,std,min,25%,50%,75%,max
0,clients,age,"400,162.0000",46.4881,43.8712,"-7,491.0000",33.0000,45.0000,59.0000,"1,901.0000"
1,products,netto,"43,035.0000",0.5370,8.2744,0.0000,0.1499,0.3053,0.5000,"1,150.0000"
2,purchases,regular_points_received,"45,786,568.0000",8.0499,12.6850,0.0000,1.3809,3.7538,10.3365,"2,399.0000"
3,purchases,express_points_received,"45,786,568.0000",0.0608,2.4262,0.0000,0.0000,0.0000,0.0000,300.0000
4,purchases,regular_points_spent,"45,786,568.0000",-5.3126,36.0365,"-5,066.0000",0.0000,0.0000,0.0000,0.0000
5,purchases,express_points_spent,"45,786,568.0000",-0.3181,3.2880,-300.0000,0.0000,0.0000,0.0000,0.0000
6,purchases,purchase_sum,"45,786,568.0000",777.5215,796.5350,0.0000,286.0927,539.0399,976.1688,"35,149.0400"
7,purchases,product_quantity,"45,786,568.0000",1.2472,3.1376,0.0000,1.0000,1.0000,1.0000,"14,941.0000"
8,purchases,trn_sum_from_iss,"45,786,568.0000",73.4884,87.5398,0.0000,30.0177,50.9557,89.7331,"35,149.0000"
9,purchases,trn_sum_from_red,"3,043,356.0000",76.7741,84.2711,0.0000,31.2326,54.5165,94.4023,"8,789.0000"


In [19]:
missing_profiles = []

for table_name, relation_name in RELATIONS.items():
    columns = [
        row[0]
        for row in duckdb_connection.execute(
            f"DESCRIBE {relation_name}"
        ).fetchall()
    ]

    expressions = [
        f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
        for column in columns
    ]

    result = duckdb_connection.execute(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            {", ".join(expressions)}
        FROM {relation_name}
        """
    ).df().iloc[0]

    total_rows = int(result["total_rows"])

    for column in columns:
        null_count = int(result[column])
        required = column in REQUIRED_NON_NULL.get(table_name, set())

        missing_profiles.append({
            "table": table_name,
            "column": column,
            "null_count": null_count,
            "null_rate": null_count / total_rows if total_rows else np.nan,
            "required_non_null": required,
        })

        if required and null_count:
            validation_issues.append(
                f"{table_name}.{column}: {null_count} unexpected null values"
            )

missing_profile = pd.DataFrame(missing_profiles)

display(
    missing_profile.sort_values(
        ["null_rate", "table", "column"],
        ascending=[False, True, True],
    )
)

,table,column,null_count,null_rate,required_non_null
28,purchases,trn_sum_from_red,42743212,0.9335,False
11,products,brand_id,5200,0.1208,False
2,clients,first_redeem_date,35469,0.0886,False
10,products,segment_id,1572,0.0365,False
12,products,vendor_id,34,0.0008,False
6,products,level_1,3,0.0001,False
7,products,level_2,3,0.0001,False
8,products,level_3,3,0.0001,False
9,products,level_4,3,0.0001,False
13,products,netto,3,0.0001,False


In [20]:
MAX_FULL_CATEGORIES = 20
TOP_CATEGORIES = 10

categorical_profiles = []
categorical_counts = {}

for table_name, columns in CATEGORICAL_COLUMNS.items():
    relation_name = RELATIONS[table_name]

    for column in columns:
        n_unique = duckdb_connection.execute(
            f"""
            SELECT COUNT(DISTINCT "{column}")
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            """
        ).fetchone()[0]

        limit = "" if n_unique <= MAX_FULL_CATEGORIES else f"LIMIT {TOP_CATEGORIES}"

        counts = duckdb_connection.execute(
            f"""
            SELECT
                "{column}" AS value,
                COUNT(*) AS count,
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            GROUP BY "{column}"
            ORDER BY count DESC
            {limit}
            """
        ).df()

        categorical_profiles.append({
            "table": table_name,
            "column": column,
            "n_unique": n_unique,
        })

        categorical_counts[(table_name, column)] = counts

display(pd.DataFrame(categorical_profiles))

for (table_name, column), counts in categorical_counts.items():
    print(f"\n{table_name}.{column}")
    display(counts)

,table,column,n_unique
0,clients,gender,3
1,products,level_1,3
2,products,level_2,42
3,products,level_3,201
4,products,level_4,790
5,products,segment_id,116
6,products,brand_id,4296
7,products,vendor_id,3193
8,purchases,store_id,13882



clients.gender


,value,count
0,U,185706
1,F,147649
2,M,66807



products.level_1


,value,count
0,e344ab2e71,22183
1,c3d3a8e8c6,16573
2,ec62ce61e3,4279


...

,value,count
0,0d6f137fb6,4344
1,4da2dc345f,3071
2,b06ace74de,385
3,ab230258e9,268
4,63ba6b7a61,136
5,7a282015f3,123
6,a548b9f2b8,116
7,74251e93eb,111
8,8188d00160,108
9,aa73f98d68,106



products.vendor_id


,value,count
0,43acd80c1a,1514
1,63243765ed,349
2,c4e167b91e,331
3,83f98e6dc3,328
4,4f276b13c1,323
5,e6af81215a,300
6,addfbe3485,224
7,41aa9501e1,223
8,3034fb4c4a,211
9,ef3b92f068,194



purchases.store_id


,value,count
0,cfbbd53ab7,18984
1,1f41964607,17436
2,a87bebd240,16833
3,f7390207ef,15856
4,3159cd57ba,15669
5,e3bf88fabf,15528
6,fac91d76e3,15219
7,c37dad9a51,15172
8,b61c786803,15070
9,dfb94aca9d,15051


In [21]:
product_binary_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE is_own_trademark IS NOT NULL
              AND is_own_trademark NOT IN (0, 1)
        ) AS invalid_is_own_trademark,
        COUNT(*) FILTER (
            WHERE is_alcohol IS NOT NULL
              AND is_alcohol NOT IN (0, 1)
        ) AS invalid_is_alcohol
    FROM products_raw
    """
).df()

display(product_binary_check)

if product_binary_check.iloc[0].sum() > 0:
    validation_issues.append("products contains invalid binary values")

,invalid_is_own_trademark,invalid_is_alcohol
0,0,0


In [22]:
range_checks = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE age IS NOT NULL AND age < 0
        ) AS negative_age
    FROM clients_raw
    """
).df()

product_range_checks = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE netto IS NOT NULL AND netto < 0
        ) AS negative_netto
    FROM products_raw
    """
).df()

display(range_checks)
display(product_range_checks)

if range_checks.iloc[0, 0] > 0:
    validation_issues.append("clients.age contains negative values")

if product_range_checks.iloc[0, 0] > 0:
    validation_issues.append("products.netto contains negative values")

,negative_age
0,96


,negative_netto
0,0


In [23]:
datetime_profiles = []

for table_name, columns in DATETIME_COLUMNS.items():
    relation_name = RELATIONS[table_name]

    for column in columns:
        min_date, max_date, future_count = duckdb_connection.execute(
            f"""
            SELECT
                MIN("{column}"),
                MAX("{column}"),
                COUNT(*) FILTER (
                    WHERE "{column}" > CURRENT_TIMESTAMP
                )
            FROM {relation_name}
            """
        ).fetchone()

        datetime_profiles.append({
            "table": table_name,
            "column": column,
            "min": min_date,
            "max": max_date,
            "future_count": future_count,
        })

        if future_count:
            validation_issues.append(
                f"{table_name}.{column}: {future_count} future timestamps"
            )

display(pd.DataFrame(datetime_profiles))

,table,column,min,max,future_count
0,clients,first_issue_date,2017-04-04 18:24:18,2019-03-15 21:50:56,0
1,clients,first_redeem_date,2017-04-11 09:42:20,2019-11-20 01:14:10,0
2,purchases,transaction_datetime,2018-11-21 21:02:33,2019-03-18 23:40:03,0


In [24]:
client_date_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE first_issue_date IS NOT NULL
              AND first_redeem_date IS NOT NULL
              AND first_redeem_date < first_issue_date
        ) AS redeem_before_issue
    FROM clients_raw
    """
).df()

display(client_date_check)

if client_date_check.iloc[0, 0] > 0:
    validation_issues.append(
        "clients contains first_redeem_date earlier than first_issue_date"
    )

,redeem_before_issue
0,536


## Structural Data Quality Summary

* The `clients`, `products`, and `uplift_train` tables all have unique identifiers. No duplicate or null keys were found in any of the three tables.

* The two main variables in `uplift_train`, `treatment_flg` and `target`, have no null values and contain only values from `{0, 1}`. The joint distribution of treatment and target is:

  * `treatment_flg = 0, target = 0`: 39,695 customers (19.84%)
  * `treatment_flg = 0, target = 1`: 60,363 customers (30.18%)
  * `treatment_flg = 1, target = 0`: 36,342 customers (18.17%)
  * `treatment_flg = 1, target = 1`: 63,639 customers (31.81%)

  Both treatment and control groups have a large number of observations and contain both outcome `0` and `1`. Therefore, the data has enough basic variation to continue with treatment-effect analysis.

* Referential-integrity checks found no orphan records.

* Numeric profiling found several values that need further investigation:

  * `clients.age` ranges from `-7,491` to `1,901`, with 96 negative values and several unusually large values. These records are investigated separately below before defining a cleaning rule.
  * `products.netto` has a maximum value of `1,150`, while the median is only around `0.3053`.
  * `purchases.product_quantity` has both the median and 75th percentile equal to `1`, but the maximum reaches `14,941`.
  * Some variables, such as `regular_points_received`, `purchase_sum`, `trn_sum_from_iss`, and `trn_sum_from_red`, also contain values that are much larger than most of the distribution.

  These values cannot be considered errors based only on descriptive statistics. The units and business meaning of some raw columns are not fully documented, so their validity needs to be checked first.

* Missing-value profiling shows that several attributes have substantial missingness:

  * `purchases.trn_sum_from_red`: 42,743,212 missing values (93.35%);
  * `products.brand_id`: 5,200 (12.08%);
  * `clients.first_redeem_date`: 35,469 (8.86%);
  * `products.segment_id`: 1,572 (3.65%);
  * `vendor_id`, `level_1`–`level_4`, and `netto` have only a small amount of missing data.

  These missing values should not automatically be treated as data errors. Some of them may represent valid business states, so their meaning should be checked based on the definition of each column.

* The two binary product flags, `is_own_trademark` and `is_alcohol`, contain only values from `{0, 1}`.

* Datetime profiling shows:

  * `transaction_datetime`: `2018-11-21` to `2019-03-18`;
  * `first_issue_date`: `2017-04-04` to `2019-03-15`;
  * `first_redeem_date`: `2017-04-11` to `2019-11-20`.

  No timestamps were found to be later than the notebook run date. However, the validity of these time variables for feature engineering still needs to be checked against the campaign/treatment cutoff to avoid using information from the future.

* The relationship between the two dates in `clients` shows 536 cases where `first_redeem_date < first_issue_date`. If the current understanding of these two columns is correct, this is a domain inconsistency. Since the dataset does not provide detailed definitions for every raw column, the column semantics should be confirmed before deciding how to handle these records.

Overall, the **structural integrity of the dataset is good**, but two main areas still need to be clarified before feature engineering:

1. **Value-quality issues**: invalid `age` values, unusual date relationships, missing values, and some extreme numeric values.
2. **Unverified data semantics**: especially the grain of `purchases`, the meaning of `purchase_sum`, loyalty-point columns, `trn_sum_from_iss`, `trn_sum_from_red`, and several customer/product attributes.

The next step is to **investigate the data semantics and the identified quality issues** before making any cleaning or feature-engineering decisions.

## Investigation of Value-Quality Issues

The structural checks identified several values that require closer investigation before cleaning or feature engineering.

An unusual value is not automatically treated as a data error. For each issue, the investigation follows the same process:

1. Define the expected value or relationship based on the current understanding of the data.
2. Separate reference records from records that violate or fall outside that expectation.
3. Examine whether the unusual records are isolated, random, repeated, or systematic.
4. Determine whether the issue is clearly invalid, potentially meaningful, plausible but extreme, or still unresolved.
5. Make a cleaning decision only after the evidence is understood.

The investigation focuses on:

- implausible customer ages;
- inconsistent customer dates;
- missing values;
- extreme numeric values.



### Customer Age

`age` is expected to represent customer age in years.

The dataset does not provide an official valid-age range. Therefore, the analysis does not immediately classify every unusually low or high age as an error.

For investigation, `13–100` is used as a working reference range:

- `13–100`: reference range for comparison;
- `< 13`: unusually low and requires investigation;
- `101–120`: unusually high but not automatically impossible;
- `> 120` or `< 0`: clearly implausible if the field truly represents age in years.

This range is used only to identify records for investigation, not as a final cleaning rule.

In [25]:
age_band_profile = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN age < 0 THEN 'Negative age'
            WHEN age < 13 THEN 'Age 0-12'
            WHEN age <= 100 THEN 'Age 13-100'
            WHEN age <= 120 THEN 'Age 101-120'
            ELSE 'Age >120'
        END AS age_group,
        COUNT(*) AS clients,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS pct_clients,
        MIN(age) AS min_age,
        MAX(age) AS max_age
    FROM clients_raw
    GROUP BY 1
    ORDER BY 1
    """
).df()

display(age_band_profile)

,age_group,clients,pct_clients,min_age,max_age
0,Age 0-12,497,0.1242,0,12
1,Age 101-120,832,0.2079,101,119
2,Age 13-100,398520,99.5897,13,100
3,Age >120,217,0.0542,121,1901
4,Negative age,96,0.0240,-7491,-1


The first step is to determine where the unusual values occur relative to the working reference range.

This separates clearly implausible values from values that are merely unusual. In particular, values between `101` and `120` should not be treated in the same way as negative ages or ages in the hundreds or thousands.

The next step is to inspect the actual suspicious values and determine whether they appear randomly or follow repeated patterns.

In [26]:
suspicious_age_counts = duckdb_connection.execute(
    """
    SELECT
        age,
        COUNT(*) AS clients
    FROM clients_raw
    WHERE age < 13
       OR age > 100
    GROUP BY age
    ORDER BY clients DESC, age
    """
).df()

print("Distinct suspicious age values:",suspicious_age_counts["age"].nunique())

print("Customers with suspicious age values:",suspicious_age_counts["clients"].sum())

age_display = suspicious_age_counts.head(30)

styled = style_table(
    age_display,
    formats={
        "age": "{:,.0f}",
        "clients": "{:,.0f}",
    },
)

repeated_idx = age_display.index[
    age_display["age"].isin([115, 119])
]

styled = paint(
    styled,
    repeated_idx,
    ["age", "clients"],
    "blue",
)

impossible_idx = age_display.index[
    (age_display["age"] < 0)
    | (age_display["age"] > 120)
]

styled = paint(
    styled,
    impossible_idx,
    ["age"],
    "red",
)

display(styled)

Distinct suspicious age values: 203
Customers with suspicious age values: 1642


age,clients
115,405
119,397
12,93
11,84
10,61
9,51
1,48
0,42
8,33
2,28


In [27]:
suspicious_age_concentration = (
    suspicious_age_counts
    .sort_values("clients", ascending=False)
    .reset_index(drop=True)
)

suspicious_age_concentration["cumulative_clients"] = (suspicious_age_concentration["clients"].cumsum())

total_suspicious_clients = suspicious_age_concentration["clients"].sum()

suspicious_age_concentration["cumulative_share"] = (
    suspicious_age_concentration["cumulative_clients"] / total_suspicious_clients
)

age_concentration_display = suspicious_age_concentration.head(20)

styled = style_table(
    age_concentration_display,
    formats={
        "age": "{:,.0f}",
        "clients": "{:,.0f}",
        "cumulative_clients": "{:,.0f}",
        "cumulative_share": "{:.2%}",
    },
)

focus_idx = [
    age_concentration_display.index[4],
    age_concentration_display.index[9],
    age_concentration_display.index[19],
]

styled = paint(
    styled,
    focus_idx,
    ["cumulative_share"],
    "blue",
)

display(styled)

age,clients,cumulative_clients,cumulative_share
115,405,405,24.67%
119,397,802,48.84%
12,93,895,54.51%
11,84,979,59.62%
10,61,"1,040",63.34%
9,51,"1,091",66.44%
1,48,"1,139",69.37%
0,42,"1,181",71.92%
8,33,"1,214",73.93%
2,28,"1,242",75.64%


In [28]:
age_concentration_summary = pd.DataFrame(
    {
        "top_values": [1, 5, 10, 20],
        "affected_clients": [
            suspicious_age_concentration.head(n)["clients"].sum()
            for n in [1, 5, 10, 20]
        ],
    }
)

age_concentration_summary["share_of_suspicious_clients"] = (
    age_concentration_summary["affected_clients"] / total_suspicious_clients
)

display(age_concentration_summary)

,top_values,affected_clients,share_of_suspicious_clients
0,1,405,0.2467
1,5,1040,0.6334
2,10,1242,0.7564
3,20,1331,0.8106


In [29]:
age_uplift_impact = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN c.age BETWEEN 13 AND 100
                THEN 'Reference age'
            ELSE 'Suspicious age'
        END AS age_status,

        COUNT(*) AS clients,

        COUNT(*) FILTER (
            WHERE u.client_id IS NOT NULL
        ) AS uplift_train_clients,

        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE u.client_id IS NOT NULL)
            / COUNT(*),
            2
        ) AS pct_in_uplift_train

    FROM clients_raw AS c

    LEFT JOIN uplift_train_raw AS u
        USING (client_id)

    GROUP BY 1
    ORDER BY 1
    """
).df()

display(age_uplift_impact)

,age_status,clients,uplift_train_clients,pct_in_uplift_train
0,Reference age,398520,199232,49.9900
1,Suspicious age,1642,807,49.1500


### Age Data Quality Investigation

Most customers have an age between `13–100`: **398,520 / 400,162 customers (99.59%)**.

There are **1,642 customers (0.41%)** outside this range:

* 497 customers have an age between `0–12`
* 832 customers have an age between `101–120`
* 217 customers have an age above `120`
* 96 customers have a negative age

The unusual values are not just a few isolated cases. Some of them appear repeatedly:

* `age = 115`: 405 customers
* `age = 119`: 397 customers
* The top 5 unusual values account for 63.34% of affected customers
* The top 10 account for 75.64%
* The top 20 account for 81.06%

There are also clearly unrealistic values such as `944`, `949`, `952`, `-943`, and `-958`, with the full range going from `-7,491` to `1,901`.

This suggests that the `age` issue is not limited to a few individual input errors. Some values may have been recorded or encoded incorrectly. However, based on the available data, we cannot reliably determine the actual age of these customers.

Among the 1,642 customers with an age outside the `13–100` range, **807 are included in `uplift_train`**, so this issue also directly affects the dataset used for modeling.

### Handling Decision

To avoid using unreliable age values in the model:

* `13 <= age <= 100`: keep the original value.
* `age < 13` or `age > 100`: convert to `unknown`.
* Do not remove customers just because their age is invalid.

The `13–100` range is used as a practical feature-engineering rule for this dataset, not as an official age range defined by RetailHero.

Converting values outside this range to `unknown` allows us to keep the other customer information without guessing or replacing the age with a value that has no clear evidence behind it.


### Customer Date Consistency

Based on the current interpretation of the customer fields:

- `first_issue_date` represents the date the customer first received or activated the loyalty account/card;
- `first_redeem_date` represents the customer's first redemption date.

Under this interpretation, the expected relationship is:

`first_issue_date <= first_redeem_date`

A missing `first_redeem_date` may be valid if the customer has never redeemed points.

The structural checks found 536 customers where `first_redeem_date < first_issue_date`. These records are investigated to determine whether the difference is small and potentially caused by timestamp or recording issues, or whether it follows a larger systematic pattern.

In [30]:
date_relation_profile = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN first_redeem_date IS NULL
                THEN 'Missing first_redeem_date'
            WHEN first_redeem_date < first_issue_date
                THEN 'Redeem before issue'
            ELSE 'Expected ordering'
        END AS date_status,
        COUNT(*) AS clients,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS pct_clients
    FROM clients_raw
    GROUP BY 1
    ORDER BY clients DESC
    """
).df()

display(date_relation_profile)

,date_status,clients,pct_clients
0,Expected ordering,364157,91.0024
1,Missing first_redeem_date,35469,8.8637
2,Redeem before issue,536,0.1339


In [31]:
redeem_before_issue_summary = duckdb_connection.execute(
    """
    WITH inconsistent_dates AS (
        SELECT
            client_id,
            first_issue_date,
            first_redeem_date,
            date_diff(
                'day',
                first_redeem_date,
                first_issue_date
            ) AS days_before_issue
        FROM clients_raw
        WHERE first_redeem_date < first_issue_date
    )

    SELECT
        COUNT(*) AS clients,
        MIN(days_before_issue) AS min_days,
        quantile_cont(days_before_issue, 0.25) AS p25_days,
        quantile_cont(days_before_issue, 0.50) AS median_days,
        quantile_cont(days_before_issue, 0.75) AS p75_days,
        quantile_cont(days_before_issue, 0.95) AS p95_days,
        MAX(days_before_issue) AS max_days
    FROM inconsistent_dates
    """
).df()

display(redeem_before_issue_summary)

,clients,min_days,p25_days,median_days,p75_days,p95_days,max_days
0,536,0,0.0000,0.0000,0.0000,0.0000,43


In [32]:
redeem_before_issue_bands = duckdb_connection.execute(
    """
    WITH inconsistent_dates AS (
        SELECT
            date_diff(
                'day',
                first_redeem_date,
                first_issue_date
            ) AS days_before_issue
        FROM clients_raw
        WHERE first_redeem_date < first_issue_date
    )

    SELECT
        CASE
            WHEN days_before_issue = 0 THEN 'Same day'
            WHEN days_before_issue <= 7 THEN '1-7 days'
            WHEN days_before_issue <= 30 THEN '8-30 days'
            WHEN days_before_issue <= 180 THEN '31-180 days'
            WHEN days_before_issue <= 365 THEN '181-365 days'
            ELSE '>365 days'
        END AS difference_group,
        COUNT(*) AS clients,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_inconsistent
    FROM inconsistent_dates
    GROUP BY 1
    ORDER BY
        MIN(days_before_issue)
    """
).df()

display(redeem_before_issue_bands)

,difference_group,clients,pct_inconsistent
0,Same day,535,99.8100
1,31-180 days,1,0.1900


In [33]:
redeem_before_issue_sample = duckdb_connection.execute(
    """
    SELECT
        client_id,
        first_issue_date,
        first_redeem_date,
        date_diff(
            'day',
            first_redeem_date,
            first_issue_date
        ) AS days_before_issue
    FROM clients_raw
    WHERE first_redeem_date < first_issue_date
    ORDER BY days_before_issue DESC
    LIMIT 20
    """
).df()

styled = style_table(
    redeem_before_issue_sample,
    formats={
        "days_before_issue": "{:,.0f}",
    },
)

inconsistent_idx = redeem_before_issue_sample.index[
    redeem_before_issue_sample["days_before_issue"] > 0
]

styled = paint(
    styled,
    inconsistent_idx,
    ["days_before_issue"],
    "red",
)

display(styled)


client_id,first_issue_date,first_redeem_date,days_before_issue
01440dcc9f,2018-11-09 13:50:09,2018-09-27 17:35:20,43
bd664f4c40,2018-09-01 20:58:46,2018-09-01 20:58:31,0
bd2fa51783,2019-02-21 18:02:36,2019-02-21 18:02:15,0
be74fdc19f,2018-11-17 21:37:17,2018-11-17 21:37:00,0
bd7a02ea6b,2019-02-08 15:12:07,2019-02-08 15:12:00,0
bdb87a7a76,2018-12-19 19:20:47,2018-12-19 19:20:40,0
be47383cb5,2019-03-10 18:35:20,2019-03-10 18:35:11,0
c40b667d3b,2018-09-28 19:17:17,2018-09-28 19:17:09,0
bf6dc0d598,2018-11-28 19:09:12,2018-11-28 19:08:28,0
bfe4de562b,2018-11-29 15:30:03,2018-11-29 15:29:53,0


In [34]:
date_issue_uplift_impact = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS inconsistent_clients,
        COUNT(*) FILTER (
            WHERE u.client_id IS NOT NULL
        ) AS uplift_train_clients,
        ROUND(
            100.0
            * COUNT(*) FILTER (WHERE u.client_id IS NOT NULL)
            / COUNT(*),
            2
        ) AS pct_in_uplift_train
    FROM clients_raw AS c

    LEFT JOIN uplift_train_raw AS u
        USING (client_id)

    WHERE c.first_redeem_date < c.first_issue_date
    """
).df()

display(date_issue_uplift_impact)

,inconsistent_clients,uplift_train_clients,pct_in_uplift_train
0,536,245,45.7100


### Date Relationship Check

* **Overall distribution (across 400,162 customers):**

  * **91.00% (364,157 customers):** The first issue date is before or equal to the first redeem date (`first_issue_date <= first_redeem_date`) — these cases are valid.
  * **8.86% (35,469 customers):** The first redeem date is missing (`first_redeem_date` is null).
  * **0.13% (536 customers):** The first redeem date is earlier than the first issue date (`first_redeem_date < first_issue_date`) — these cases are considered unusual.

* **Detailed analysis of the 536 unusual cases:**

  * **535 cases (99.81%):** The two dates are on the **same day**, with only a difference of a few to a few dozen seconds (for example, the card was issued at 20:58:46 but first redeemed at 20:58:31). These are unlikely to be serious data errors and may be caused by the system recording events in a slightly different order on the same day.
  * **1 case:** The difference is **43 days**, with the redeem date occurring much earlier than the issue date. This cannot be easily explained by a small timestamp recording issue.
  * **Impact:** 245 affected customers are included in `uplift_train`, which is a very small proportion of the training data and is unlikely to have a meaningful overall impact.

### Handling Decision

1. **For the 535 same-day cases:** Treat `first_issue_date` and `first_redeem_date` as occurring on the same day when creating features. These customers do not need to be removed.
2. **For the single 43-day inconsistency:** Since the data is contradictory and we cannot determine which date is correct, convert the affected value to **`unknown` (missing)** during feature engineering instead of manually changing it.
3. **For customers with missing `first_redeem_date`:** These cases will be investigated and handled separately in the missing-value analysis.


### Product `netto`

`netto` is a numeric product attribute. The dataset does not provide enough documentation to define its exact unit or a valid upper limit.

The only clear value rule available from the data is:

- `netto` should not be negative.

The structural checks found no negative values, but the distribution is strongly skewed: the median is about `0.3053`, while the maximum reaches `1,150`.

Because the unit and business meaning are not fully documented, large values are not automatically treated as errors. The investigation first checks how the upper tail is distributed and whether the extreme values are isolated or follow a repeated pattern.

In [35]:
netto_distribution = duckdb_connection.execute(
    """
    SELECT
        COUNT(netto) AS products,
        MIN(netto) AS min,
        quantile_cont(netto, 0.25) AS p25,
        quantile_cont(netto, 0.50) AS median,
        quantile_cont(netto, 0.75) AS p75,
        quantile_cont(netto, 0.95) AS p95,
        quantile_cont(netto, 0.99) AS p99,
        quantile_cont(netto, 0.999) AS p999,
        MAX(netto) AS max
    FROM products_raw
    WHERE netto IS NOT NULL
    """
).df()

display(netto_distribution)

,products,min,p25,median,p75,p95,p99,p999,max
0,43035,0.0000,0.1500,0.3000,0.5000,1.0000,1.5586,5.0000,"1,150.0000"


In [36]:
netto_tail_summary = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(netto, 0.999) AS p999
        FROM products_raw
        WHERE netto IS NOT NULL
    )
    SELECT
        p999 AS investigation_threshold,
        COUNT(*) FILTER (
            WHERE netto >= p999
        ) AS extreme_products,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE netto >= p999)
            / COUNT(*),
            4
        ) AS pct_products
    FROM products_raw
    CROSS JOIN threshold
    WHERE netto IS NOT NULL
    GROUP BY p999
    """
).df()

display(netto_tail_summary)

,investigation_threshold,extreme_products,pct_products
0,5.0000,99,0.2300


In [37]:
extreme_netto_values = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(netto, 0.999) AS p999
        FROM products_raw
        WHERE netto IS NOT NULL
    )
    SELECT
        netto,
        COUNT(*) AS products
    FROM products_raw
    CROSS JOIN threshold
    WHERE netto >= p999
    GROUP BY netto
    ORDER BY netto DESC
    """
).df()

netto_display = extreme_netto_values.head(30)

styled = style_table(
    netto_display,
    formats={
        "netto": "{:,.4f}",
        "products": "{:,.0f}",
    },
)

max_idx = netto_display.index[
    netto_display["netto"] == netto_display["netto"].max()
]

styled = paint(
    styled,
    max_idx,
    ["netto"],
    "blue",
)

display(styled)


netto,products
"1,150.0000",1
600.0000,2
430.0000,1
400.0000,1
350.0000,1
316.8000,2
260.0000,1
259.2000,2
130.0000,1
80.0000,1


In [38]:
extreme_netto_products = duckdb_connection.execute(
    """
    SELECT
        product_id,
        level_1,
        level_2,
        level_3,
        level_4,
        segment_id,
        brand_id,
        vendor_id,
        netto
    FROM products_raw
    WHERE netto IS NOT NULL
    ORDER BY netto DESC
    LIMIT 30
    """
).df()

styled = style_table(
    extreme_netto_products,
    formats={
        "segment_id": "{:,.0f}",
        "netto": "{:,.4f}",
    },
)

focus_idx = extreme_netto_products.head(5).index

styled = paint(
    styled,
    focus_idx,
    ["netto"],
    "blue",
)

display(styled)


product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto
ec9077783d,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259,—,2a74768430,"1,150.0000"
98528c8b99,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259,—,d52d937443,600.0000
2fbd8c3d9c,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,259,—,d52d937443,600.0000
61a26b3d4b,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105,4da2dc345f,7cea9f2605,430.0000
0020caa486,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105,4da2dc345f,7cea9f2605,400.0000
b46720cd94,c3d3a8e8c6,f93982269d,0bcfc6519b,fc32a80dcd,259,85606f387b,2a74768430,350.0000
44f2e529ac,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212,563ceeb1d9,4503f4981c,316.8000
42511cfd14,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212,563ceeb1d9,4503f4981c,316.8000
50e2016e67,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,105,4da2dc345f,7cea9f2605,260.0000
d07c11c602,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,212,563ceeb1d9,4503f4981c,259.2000


In [39]:
extreme_netto_purchase_impact = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(netto, 0.999) AS p999
        FROM products_raw
        WHERE netto IS NOT NULL
    ),
    extreme_products AS (
        SELECT p.product_id
        FROM products_raw AS p
        CROSS JOIN threshold
        WHERE p.netto >= threshold.p999
    )
    SELECT
        COUNT(DISTINCT ep.product_id) AS extreme_products,
        COUNT(p.product_id) AS purchase_rows,
        COUNT(DISTINCT p.transaction_id) AS transactions,
        COUNT(DISTINCT p.client_id) AS customers
    FROM extreme_products AS ep
    LEFT JOIN purchases_raw AS p
        ON ep.product_id = p.product_id
    """
).df()

display(extreme_netto_purchase_impact)

,extreme_products,purchase_rows,transactions,customers
0,99,151841,149623,67076


In [40]:
extreme_netto_purchase_frequency = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(netto, 0.999) AS p999
        FROM products_raw
        WHERE netto IS NOT NULL
    )
    SELECT
        pr.product_id,
        pr.netto,
        COUNT(p.product_id) AS purchase_rows,
        COUNT(DISTINCT p.transaction_id) AS transactions,
        COUNT(DISTINCT p.client_id) AS customers
    FROM products_raw AS pr
    CROSS JOIN threshold
    LEFT JOIN purchases_raw AS p
        ON pr.product_id = p.product_id
    WHERE pr.netto >= threshold.p999
    GROUP BY
        pr.product_id,
        pr.netto
    ORDER BY purchase_rows DESC, pr.netto DESC
    LIMIT 30
    """
).df()

styled = style_table(
    extreme_netto_purchase_frequency,
    formats={
        "netto": "{:,.4f}",
        "purchase_rows": "{:,.0f}",
        "transactions": "{:,.0f}",
        "customers": "{:,.0f}",
    },
)

top_frequency_idx = extreme_netto_purchase_frequency.head(5).index

styled = paint(
    styled,
    top_frequency_idx,
    ["purchase_rows", "transactions", "customers"],
    "blue",
)

display(styled)


product_id,netto,purchase_rows,transactions,customers
d60345e432,5.0000,"34,627","34,627","13,159"
851e5a9155,5.0000,"28,593","28,593","19,314"
de6f428f24,5.0000,"12,423","12,423","9,702"
80afa63494,5.0000,"7,127","7,127","3,591"
7ce1ae3fd1,5.0000,"6,774","6,774","3,258"
d640f1d346,5.0000,"5,805","5,805","3,348"
f47cf459d2,24.0000,"4,842","4,842","2,375"
f6e1484ec9,5.0000,"4,747","4,747","2,840"
0b610b0340,5.0000,"2,734","2,734","1,261"
2c48e69f2a,5.0000,"2,642","2,642","1,421"


## Netto Data Quality Investigation

### 1. Data Distribution

Most products have a very small `netto` value:

* **Median:** around `0.30`.
* **95%** of products have `netto <= 1.00`.
* **99%** of products have `netto <= 1.56`.
* **99.9%** of products have `netto <= 5.00`.

Only a very small group of products — 99 products, or **0.23%** — have `netto >= 5.00`, with the maximum value reaching **1,150**.

*Note:* The `5.00` threshold is only used to identify the upper tail of the distribution for further investigation. It is **not** considered a limit for determining whether a value is valid or invalid.

### 2. Findings

After a closer look, the large values are usually concentrated within specific product groups. For example, values such as `1150` and `600` appear within the same group, while `430`, `400`, and `260` appear in another group. Similarly, `316.8` and `259.2` are also found within the same product group.

This suggests that large `netto` values are related to the characteristics of specific product types rather than random data-entry errors.

### 3. Handling Decision

Since the dataset does not clearly define the unit or valid range of `netto`, we should not set an arbitrary upper limit and convert large values to missing values.

Therefore:

* Keep all `netto >= 0` values unchanged.
* Do not treat `netto > 5.00` as an error.
* Missing `netto` values will be investigated and handled separately.


### Product Quantity

`product_quantity` represents the quantity recorded for a product in a purchase row.

The structural profile shows that most values are small:

- Median = `1`;
- 75th percentile = `1`;
- Maximum = `14,941`.

A large quantity is not automatically treated as an error because the dataset does not provide an official upper limit and some purchases may legitimately contain multiple units.

The investigation therefore checks:

- Whether non-positive quantities exist;
- How quickly the upper tail increases;
- Whether very large quantities are isolated or repeated;
- Whether extreme values are concentrated in particular products, transactions, or customers.

In [41]:
product_quantity_distribution = duckdb_connection.execute(
    """
    SELECT
        COUNT(product_quantity) AS purchase_rows,
        MIN(product_quantity) AS min,
        quantile_cont(product_quantity, 0.25) AS p25,
        quantile_cont(product_quantity, 0.50) AS median,
        quantile_cont(product_quantity, 0.75) AS p75,
        quantile_cont(product_quantity, 0.95) AS p95,
        quantile_cont(product_quantity, 0.99) AS p99,
        quantile_cont(product_quantity, 0.999) AS p999,
        quantile_cont(product_quantity, 0.9999) AS p9999,
        MAX(product_quantity) AS max
    FROM purchases_raw
    WHERE product_quantity IS NOT NULL
    """
).df()

display(product_quantity_distribution)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,purchase_rows,min,p25,median,p75,p95,p99,p999,p9999,max
0,45786568,0.0000,1.0000,1.0000,1.0000,3.0000,5.0000,11.0000,30.0000,"14,941.0000"


In [42]:
product_quantity_nonpositive = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN product_quantity < 0 THEN 'Negative'
            WHEN product_quantity = 0 THEN 'Zero'
            ELSE 'Positive'
        END AS quantity_status,
        COUNT(*) AS purchase_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            4
        ) AS pct_rows
    FROM purchases_raw
    WHERE product_quantity IS NOT NULL
    GROUP BY 1
    ORDER BY 1
    """
).df()

display(product_quantity_nonpositive)

,quantity_status,purchase_rows,pct_rows
0,Positive,43079290,94.0872
1,Zero,2707278,5.9128


In [43]:
product_quantity_tail = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(product_quantity, 0.999) AS p999
        FROM purchases_raw
        WHERE product_quantity IS NOT NULL
    )
    SELECT
        p999 AS investigation_threshold,
        COUNT(*) FILTER (
            WHERE product_quantity >= p999
        ) AS extreme_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE product_quantity >= p999)
            / COUNT(*),
            4
        ) AS pct_rows
    FROM purchases_raw
    CROSS JOIN threshold
    WHERE product_quantity IS NOT NULL
    GROUP BY p999
    """
).df()

display(product_quantity_tail)

,investigation_threshold,extreme_rows,pct_rows
0,11.0000,49186,0.1074


In [44]:
extreme_quantity_values = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(product_quantity, 0.999) AS p999
        FROM purchases_raw
        WHERE product_quantity IS NOT NULL
    )
    SELECT
        product_quantity,
        COUNT(*) AS purchase_rows
    FROM purchases_raw
    CROSS JOIN threshold
    WHERE product_quantity >= p999
    GROUP BY product_quantity
    ORDER BY product_quantity DESC
    """
).df()

quantity_display = extreme_quantity_values.head(30)

styled = style_table(
    quantity_display,
    formats={
        "product_quantity": "{:,.0f}",
        "purchase_rows": "{:,.0f}",
    },
)

# Three values classified as unreliable.
invalid_idx = quantity_display.index[
    quantity_display["product_quantity"] > 1000
]

styled = paint(
    styled,
    invalid_idx,
    ["product_quantity", "purchase_rows"],
    "red",
)

# Next-largest value, used to show the large gap.
next_value_idx = quantity_display.index[
    quantity_display["product_quantity"] == 660
]

styled = paint(
    styled,
    next_value_idx,
    ["product_quantity"],
    "blue",
)

display(styled)

product_quantity,purchase_rows
"14,941",1
"9,228",1
"9,227",1
660,1
648,1
620,1
600,1
500,3
420,1
400,2


In [45]:
extreme_quantity_rows = duckdb_connection.execute(
    """
    SELECT
        client_id,
        transaction_id,
        transaction_datetime,
        store_id,
        product_id,
        product_quantity,
        purchase_sum,
        trn_sum_from_iss,
        trn_sum_from_red
    FROM purchases_raw
    WHERE product_quantity IS NOT NULL
    ORDER BY product_quantity DESC
    LIMIT 30
    """
).df()

styled = style_table(
    extreme_quantity_rows,
    formats={
        "product_quantity": "{:,.0f}",
        "purchase_sum": "{:,.2f}",
        "trn_sum_from_iss": "{:,.2f}",
        "trn_sum_from_red": "{:,.2f}",
    },
)

invalid_idx = extreme_quantity_rows.index[
    extreme_quantity_rows["product_quantity"] > 1000
]

styled = paint(
    styled,
    invalid_idx,
    ["product_quantity"],
    "red",
)

next_value_idx = extreme_quantity_rows.index[
    extreme_quantity_rows["product_quantity"] == 660
]

styled = paint(
    styled,
    next_value_idx,
    ["product_quantity"],
    "blue",
)

display(styled)

client_id,transaction_id,transaction_datetime,store_id,product_id,product_quantity,purchase_sum,trn_sum_from_iss,trn_sum_from_red
398c509c73,160d21b04f,2018-12-31 13:12:06,2b2380f90e,ce5dddfb68,"14,941",545.00,35.00,—
e1fed74d0c,e4279a30c6,2019-01-14 14:45:32,89837f8236,e7ad2b87e1,"9,228",187.00,19.00,—
e1fed74d0c,e4279a30c6,2019-01-14 14:45:32,89837f8236,5481e41adf,"9,227",187.00,80.00,—
84db983b3b,7660acc6f1,2018-12-08 10:38:41,bedb74e9c7,47561f90e6,660,"3,300.00","3,300.00",—
d5eda4436b,482a36036c,2019-01-16 07:01:52,2160fd89b8,ee4f7132c3,648,"25,913.00","25,914.00",—
eaaebcf418,cda1c1762e,2019-03-04 09:48:56,95e25964e6,4009f09b04,620,"2,790.00","2,790.00",—
d5eda4436b,5ebb3e6359,2018-12-13 13:33:37,2160fd89b8,ee4f7132c3,600,"23,994.00","23,994.00",—
f1c52877b3,8535979e44,2018-11-26 06:20:31,4df9ddf613,4dcf79043e,500,"19,345.00","19,345.00",—
9bd5d53937,062db146e1,2019-03-15 07:06:28,c69b9cef7a,4dcf79043e,500,"18,445.00","18,445.00",—
9bd5d53937,8d43957f5c,2019-03-15 07:07:06,c69b9cef7a,4dcf79043e,500,"18,445.00","18,445.00",—


In [46]:
extreme_quantity_products = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(product_quantity, 0.999) AS p999
        FROM purchases_raw
        WHERE product_quantity IS NOT NULL
    )
    SELECT
        product_id,
        COUNT(*) AS extreme_rows,
        COUNT(DISTINCT transaction_id) AS transactions,
        COUNT(DISTINCT client_id) AS customers,
        MIN(product_quantity) AS min_extreme_quantity,
        MAX(product_quantity) AS max_extreme_quantity
    FROM purchases_raw
    CROSS JOIN threshold
    WHERE product_quantity >= p999
    GROUP BY product_id
    ORDER BY extreme_rows DESC, max_extreme_quantity DESC
    LIMIT 30
    """
).df()

styled = style_table(
    extreme_quantity_products,
    formats={
        "extreme_rows": "{:,.0f}",
        "transactions": "{:,.0f}",
        "customers": "{:,.0f}",
        "min_extreme_quantity": "{:,.0f}",
        "max_extreme_quantity": "{:,.0f}",
    },
)

top_extreme_product_idx = extreme_quantity_products.head(5).index

styled = paint(
    styled,
    top_extreme_product_idx,
    ["extreme_rows", "transactions", "customers"],
    "blue",
)

invalid_idx = extreme_quantity_products.index[
    extreme_quantity_products["max_extreme_quantity"] > 1000
]

styled = paint(
    styled,
    invalid_idx,
    ["max_extreme_quantity"],
    "red",
)

display(styled)


product_id,extreme_rows,transactions,customers,min_extreme_quantity,max_extreme_quantity
4dcf79043e,"4,371","4,371","3,089",11,500
ea27d5dc75,"1,029","1,029",217,11,300
113e3ace79,746,746,532,11,162
e29cab0243,634,634,448,11,112
ee4f7132c3,614,614,412,11,648
1c257c1a1b,495,495,205,11,210
f2293d7dfa,486,486,404,11,243
cf1a5be7fb,450,450,315,11,121
422d732e1d,400,400,350,11,149
67d2476667,364,364,267,11,82


In [47]:
extreme_quantity_customers = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT
            quantile_cont(product_quantity, 0.999) AS p999
        FROM purchases_raw
        WHERE product_quantity IS NOT NULL
    )
    SELECT
        client_id,
        COUNT(*) AS extreme_rows,
        COUNT(DISTINCT transaction_id) AS transactions,
        COUNT(DISTINCT product_id) AS products,
        MAX(product_quantity) AS max_quantity
    FROM purchases_raw
    CROSS JOIN threshold
    WHERE product_quantity >= p999
    GROUP BY client_id
    ORDER BY extreme_rows DESC, max_quantity DESC
    LIMIT 30
    """
).df()

styled = style_table(
    extreme_quantity_customers,
    formats={
        "extreme_rows": "{:,.0f}",
        "transactions": "{:,.0f}",
        "products": "{:,.0f}",
        "max_quantity": "{:,.0f}",
    },
)

top_extreme_customer_idx = extreme_quantity_customers.head(5).index

styled = paint(
    styled,
    top_extreme_customer_idx,
    ["extreme_rows", "transactions", "products"],
    "blue",
)

large_customer_quantity_idx = extreme_quantity_customers.index[
    extreme_quantity_customers["max_quantity"] >= 100
]

styled = paint(
    styled,
    large_customer_quantity_idx,
    ["max_quantity"],
    "blue",
)

display(styled)


client_id,extreme_rows,transactions,products,max_quantity
dc9477d643,238,123,28,45
58aeee1499,166,45,113,120
085b23ad91,159,152,6,68
9bd5d53937,147,92,64,500
84db983b3b,105,54,73,660
9ddf1c5829,97,57,43,81
ec251b8dbc,85,85,2,131
db7125fe49,77,71,7,81
98499231ed,75,64,6,40
e178dbfb24,60,17,24,62


In [48]:
extreme_quantity_product_profile = duckdb_connection.execute(
    """
    WITH product_quantity_stats AS (
        SELECT
            product_id,
            COUNT(*) AS purchase_rows,
            quantile_cont(product_quantity, 0.50) AS median_quantity,
            quantile_cont(product_quantity, 0.95) AS p95_quantity,
            quantile_cont(product_quantity, 0.99) AS p99_quantity,
            MAX(product_quantity) AS max_quantity,
            COUNT(*) FILTER (
                WHERE product_quantity > 30
            ) AS rows_above_30
        FROM purchases_raw
        WHERE product_quantity > 0
        GROUP BY product_id
    ),

    largest_products AS (
        SELECT product_id
        FROM product_quantity_stats
        ORDER BY max_quantity DESC
        LIMIT 30
    )

    SELECT
        s.product_id,
        p.level_1,
        p.level_2,
        p.level_3,
        p.level_4,
        p.segment_id,
        p.netto,
        s.purchase_rows,
        s.median_quantity,
        s.p95_quantity,
        s.p99_quantity,
        s.max_quantity,
        s.rows_above_30
    FROM product_quantity_stats AS s

    JOIN largest_products AS e
        USING (product_id)

    LEFT JOIN products_raw AS p
        USING (product_id)

    ORDER BY s.max_quantity DESC
    """
).df()

styled = style_table(
    extreme_quantity_product_profile,
    formats={
        "segment_id": "{:,.0f}",
        "netto": "{:,.4f}",
        "purchase_rows": "{:,.0f}",
        "median_quantity": "{:,.2f}",
        "p95_quantity": "{:,.2f}",
        "p99_quantity": "{:,.2f}",
        "max_quantity": "{:,.0f}",
        "rows_above_30": "{:,.0f}",
    },
)

invalid_idx = extreme_quantity_product_profile.index[
    extreme_quantity_product_profile["max_quantity"] > 1000
]

styled = paint(
    styled,
    invalid_idx,
    ["median_quantity", "p95_quantity", "p99_quantity"],
    "blue",
)

styled = paint(
    styled,
    invalid_idx,
    ["max_quantity"],
    "red",
)

display(styled)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

product_id,level_1,level_2,level_3,level_4,segment_id,netto,purchase_rows,median_quantity,p95_quantity,p99_quantity,max_quantity,rows_above_30
ce5dddfb68,e344ab2e71,703f4b6eb0,8a37c27a14,8587e8dfd1,119,0.1370,"10,733",1.00,2.00,3.00,"14,941",1
e7ad2b87e1,e344ab2e71,703f4b6eb0,0c37077fa0,1e98c70e32,209,0.0120,"20,609",2.00,6.00,15.00,"9,228",46
5481e41adf,c3d3a8e8c6,ad2b2e17d2,d7ca614c25,b201a329b9,214,0.1800,"7,686",1.00,2.00,4.00,"9,227",2
47561f90e6,e344ab2e71,1d2939ba1d,334b74af37,146717c1b2,18,0.0130,"2,210",1.00,2.00,3.91,660,1
ee4f7132c3,e344ab2e71,14d373dff5,39532a0f6f,e66f0cae96,92,0.4500,"27,376",2.00,8.00,15.25,648,46
4009f09b04,e344ab2e71,1d2939ba1d,334b74af37,146717c1b2,18,0.0150,"1,824,586",1.00,2.00,3.00,620,62
4dcf79043e,e344ab2e71,ed2ad1797c,c4669106da,9b12b02458,262,1.0000,"370,451",1.00,5.00,12.00,500,556
ea27d5dc75,c3d3a8e8c6,d283080a93,f976a3c5f9,5193c83f20,48,1.0000,"147,463",2.00,4.00,8.00,300,236
a755a6f2e6,e344ab2e71,703f4b6eb0,8a37c27a14,8587e8dfd1,119,0.0950,"13,811",1.00,3.00,4.00,278,4
197c432c53,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,9ea7822731,37,0.0400,"12,912",2.00,6.00,10.00,254,5


In [49]:
very_large_quantity_summary = duckdb_connection.execute(
    """
    SELECT
        product_id,
        COUNT(*) AS rows_above_30,
        COUNT(DISTINCT client_id) AS customers,
        COUNT(DISTINCT transaction_id) AS transactions,
        MIN(product_quantity) AS min_quantity,
        quantile_cont(product_quantity, 0.50) AS median_quantity,
        MAX(product_quantity) AS max_quantity
    FROM purchases_raw
    WHERE product_quantity > 30
    GROUP BY product_id
    ORDER BY max_quantity DESC
    LIMIT 30
    """
).df()

styled = style_table(
    very_large_quantity_summary,
    formats={
        "rows_above_30": "{:,.0f}",
        "customers": "{:,.0f}",
        "transactions": "{:,.0f}",
        "min_quantity": "{:,.0f}",
        "median_quantity": "{:,.2f}",
        "max_quantity": "{:,.0f}",
    },
)

invalid_idx = very_large_quantity_summary.index[
    very_large_quantity_summary["max_quantity"] > 1000
]

styled = paint(
    styled,
    invalid_idx,
    ["max_quantity"],
    "red",
)

repeated_large_idx = very_large_quantity_summary.index[
    very_large_quantity_summary["rows_above_30"] >= 100
]

styled = paint(
    styled,
    repeated_large_idx,
    ["rows_above_30", "customers", "transactions"],
    "blue",
)

display(styled)


product_id,rows_above_30,customers,transactions,min_quantity,median_quantity,max_quantity
ce5dddfb68,1,1,1,"14,941","14,941.00","14,941"
e7ad2b87e1,46,45,46,31,39.50,"9,228"
5481e41adf,2,2,2,32,"4,629.50","9,227"
47561f90e6,1,1,1,660,660.00,660
ee4f7132c3,46,27,46,33,48.00,648
4009f09b04,62,55,62,31,76.00,620
4dcf79043e,556,322,556,31,50.00,500
ea27d5dc75,236,40,236,31,52.00,300
a755a6f2e6,4,4,4,35,48.50,278
197c432c53,5,5,5,36,45.00,254


In [50]:
zero_quantity_profile = duckdb_connection.execute(
    """
    WITH product_status AS (
        SELECT
            product_id,
            COUNT(*) FILTER (
                WHERE product_quantity = 0
            ) AS zero_rows,
            COUNT(*) FILTER (
                WHERE product_quantity > 0
            ) AS positive_rows
        FROM purchases_raw
        GROUP BY product_id
    )

    SELECT
        COUNT(*) FILTER (
            WHERE zero_rows > 0
        ) AS products_with_zero,

        COUNT(*) FILTER (
            WHERE zero_rows > 0
              AND positive_rows > 0
        ) AS products_with_zero_and_positive,

        COUNT(*) FILTER (
            WHERE zero_rows > 0
              AND positive_rows = 0
        ) AS zero_only_products,

        SUM(zero_rows) AS zero_rows
    FROM product_status
    """
).df()

display(zero_quantity_profile)

,products_with_zero,products_with_zero_and_positive,zero_only_products,zero_rows
0,1330,1053,277,"2,707,278.0000"


In [51]:
zero_quantity_products = duckdb_connection.execute(
    """
    SELECT
        pu.product_id,
        pr.level_1,
        pr.level_2,
        pr.level_3,
        pr.level_4,
        pr.segment_id,
        pr.netto,

        COUNT(*) FILTER (
            WHERE pu.product_quantity = 0
        ) AS zero_rows,

        COUNT(*) FILTER (
            WHERE pu.product_quantity > 0
        ) AS positive_rows,

        COUNT(DISTINCT pu.client_id) FILTER (
            WHERE pu.product_quantity = 0
        ) AS zero_customers

    FROM purchases_raw AS pu

    LEFT JOIN products_raw AS pr
        USING (product_id)

    GROUP BY
        pu.product_id,
        pr.level_1,
        pr.level_2,
        pr.level_3,
        pr.level_4,
        pr.segment_id,
        pr.netto

    HAVING COUNT(*) FILTER (
        WHERE pu.product_quantity = 0
    ) > 0

    ORDER BY zero_rows DESC
    LIMIT 30
    """
).df()

styled = style_table(
    zero_quantity_products,
    formats={
        "segment_id": "{:,.0f}",
        "netto": "{:,.4f}",
        "zero_rows": "{:,.0f}",
        "positive_rows": "{:,.0f}",
        "zero_customers": "{:,.0f}",
    },
)

top_zero_idx = zero_quantity_products.head(5).index

styled = paint(
    styled,
    top_zero_idx,
    ["zero_rows", "zero_customers"],
    "blue",
)

display(styled)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

product_id,level_1,level_2,level_3,level_4,segment_id,netto,zero_rows,positive_rows,zero_customers
4a29330c8d,c3d3a8e8c6,034aca0659,6826908da1,dfa74aef29,3,1.0000,"263,443","29,961","138,461"
4c07cb5835,c3d3a8e8c6,034aca0659,b67737054d,97481f1fb6,104,1.0000,"131,004","22,771","77,418"
f098ee2a85,c3d3a8e8c6,034aca0659,b67737054d,0cb3a4882f,104,1.0000,"126,900","61,028","77,064"
343e841aaa,c3d3a8e8c6,034aca0659,b67737054d,3618d329f6,104,1.0000,"117,707","113,220","70,528"
15ccaa8685,c3d3a8e8c6,034aca0659,44578de904,c76a16e13e,144,1.0000,"106,671","522,239","52,722"
cbe1cd3bb3,c3d3a8e8c6,034aca0659,b67737054d,0cb3a4882f,104,1.0000,"92,459","47,621","59,170"
5cb93c9bc5,c3d3a8e8c6,034aca0659,453baf42c5,a6ccbb75cc,281,1.0000,"75,294","245,407","44,536"
dc2001d036,c3d3a8e8c6,034aca0659,453baf42c5,39fa15598b,81,1.0000,"64,317","142,152","41,409"
bf07df54e1,c3d3a8e8c6,034aca0659,6826908da1,dfa74aef29,3,1.0000,"60,519","354,226","38,077"
d860464ae1,c3d3a8e8c6,034aca0659,b67737054d,3618d329f6,104,1.0000,"53,533","36,912","38,803"


In [52]:
quantity_integer_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,

        COUNT(*) FILTER (
            WHERE product_quantity != FLOOR(product_quantity)
        ) AS fractional_rows,

        COUNT(DISTINCT product_quantity) FILTER (
            WHERE product_quantity != FLOOR(product_quantity)
        ) AS distinct_fractional_values

    FROM purchases_raw
    WHERE product_quantity IS NOT NULL
    """
).df()

display(quantity_integer_check)

,purchase_rows,fractional_rows,distinct_fractional_values
0,45786568,0,0


### Product Quantity Data Quality Check

Most `product_quantity` values are relatively small:

* **Median:** `1`;
* **99%** of purchase rows have a quantity `<= 5`;
* **99.9%** have a quantity `<= 11`.

However, there are also some very large values.

When checking the quantity by product, values ranging from a few dozen to a few hundred appear repeatedly across different transactions and customers. Therefore, not all large quantities can be considered errors.

The three largest values are `14,941`, `9,228`, and `9,227`.

These three values are very different from the usual purchase history of their respective products. Those products normally have quantities around `1–2`, but suddenly have a single transaction with a quantity in the thousands. The next largest value is only `660`.

This suggests that these three values are more likely to be unreliable data rather than normal purchase quantities.

In addition, **2,707,278 rows (5.91%)** have `product_quantity = 0`. The value `0` appears across many products and occurs frequently, so there is currently not enough evidence to treat it as a data error.

### Handling Decision

* Keep `product_quantity` values from `0` to `1000`.
* Convert `product_quantity > 1000` to **`unknown` (missing)**.
* Do not remove a transaction or customer just because of an unusual quantity value.
* Keep `product_quantity = 0` because this is a common pattern in the dataset.

The `1000` threshold is a dataset-specific rule. It is based on the large gap between the three extreme values above `9,000` and the next largest value of `660`.


### Purchase Table Grain and Column Consistency

The previous structural checks confirmed that the main IDs and table relationships are valid.

The next step is to understand how rows inside `purchases` are structured.

A single transaction can contain multiple purchase rows, so the analysis checks:

- Which columns stay the same across rows of the same transaction;
- Which columns change between product rows;
- Whether `purchase_sum` and loyalty-point fields are transaction-level values;
- Whether `trn_sum_from_iss` and `trn_sum_from_red` behave like product-line values.

This is needed to avoid counting transaction-level values multiple times when analyzing or aggregating the purchase data.

In [53]:
purchase_full_row_duplicate_check = duckdb_connection.execute(
    """
    WITH distinct_rows AS (
        SELECT DISTINCT *
        FROM purchases_raw
    )

    SELECT
        (SELECT COUNT(*) FROM purchases_raw) AS purchase_rows,
        (SELECT COUNT(*) FROM distinct_rows) AS distinct_purchase_rows
    """
).df()

purchase_full_row_duplicate_check["duplicate_purchase_rows"] = (
    purchase_full_row_duplicate_check["purchase_rows"]
    - purchase_full_row_duplicate_check["distinct_purchase_rows"]
)

display(purchase_full_row_duplicate_check)

duplicate_purchase_rows = int(
    purchase_full_row_duplicate_check.loc[0, "duplicate_purchase_rows"]
)

if duplicate_purchase_rows:
    validation_issues.append(
        f"purchases: {duplicate_purchase_rows} duplicate full rows"
    )


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,purchase_rows,distinct_purchase_rows,duplicate_purchase_rows
0,45786568,45786568,0


In [54]:
transaction_consistency = duckdb_connection.execute(
    """
    WITH transaction_profile AS (
        SELECT
            transaction_id,
            COUNT(DISTINCT client_id) AS clients,
            COUNT(DISTINCT transaction_datetime) AS datetimes,
            COUNT(DISTINCT store_id) AS stores
        FROM purchases_raw
        GROUP BY transaction_id
    )

    SELECT
        COUNT(*) AS transactions,
        COUNT(*) FILTER (WHERE clients > 1) AS multiple_client_transactions,
        COUNT(*) FILTER (WHERE datetimes > 1) AS multiple_datetime_transactions,
        COUNT(*) FILTER (WHERE stores > 1) AS multiple_store_transactions
    FROM transaction_profile
    """
).df()

display(transaction_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,multiple_client_transactions,multiple_datetime_transactions,multiple_store_transactions
0,8045201,28,28,28


In [55]:
transaction_grain_profile = duckdb_connection.execute(
    """
    WITH transaction_profile AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            COUNT(*) AS rows_per_transaction,
            COUNT(DISTINCT product_id) AS products_per_transaction

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        COUNT(*) AS transactions,

        quantile_cont(rows_per_transaction, 0.50) AS median_rows,
        quantile_cont(rows_per_transaction, 0.95) AS p95_rows,
        quantile_cont(rows_per_transaction, 0.99) AS p99_rows,
        MAX(rows_per_transaction) AS max_rows,

        quantile_cont(products_per_transaction, 0.50) AS median_products,
        quantile_cont(products_per_transaction, 0.95) AS p95_products,
        quantile_cont(products_per_transaction, 0.99) AS p99_products,
        MAX(products_per_transaction) AS max_products

    FROM transaction_profile
    """
).df()

display(transaction_grain_profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,median_rows,p95_rows,p99_rows,max_rows,median_products,p95_products,p99_products,max_products
0,8045229,4.0000,15.0000,24.0000,116,4.0000,15.0000,24.0000,116


In [56]:
transaction_field_consistency = duckdb_connection.execute(
    """
    WITH transaction_profile AS (
        SELECT
            transaction_id,

            COUNT(DISTINCT purchase_sum) AS purchase_sum_values,
            COUNT(DISTINCT regular_points_received) AS regular_received_values,
            COUNT(DISTINCT express_points_received) AS express_received_values,
            COUNT(DISTINCT regular_points_spent) AS regular_spent_values,
            COUNT(DISTINCT express_points_spent) AS express_spent_values

        FROM purchases_raw
        GROUP BY transaction_id
    )

    SELECT
        COUNT(*) AS transactions,

        COUNT(*) FILTER (
            WHERE purchase_sum_values > 1
        ) AS inconsistent_purchase_sum,

        COUNT(*) FILTER (
            WHERE regular_received_values > 1
        ) AS inconsistent_regular_points_received,

        COUNT(*) FILTER (
            WHERE express_received_values > 1
        ) AS inconsistent_express_points_received,

        COUNT(*) FILTER (
            WHERE regular_spent_values > 1
        ) AS inconsistent_regular_points_spent,

        COUNT(*) FILTER (
            WHERE express_spent_values > 1
        ) AS inconsistent_express_points_spent

    FROM transaction_profile
    """
).df()

display(transaction_field_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,inconsistent_purchase_sum,inconsistent_regular_points_received,inconsistent_express_points_received,inconsistent_regular_points_spent,inconsistent_express_points_spent
0,8045201,28,27,0,5,2


In [57]:
line_value_consistency = duckdb_connection.execute(
    """
    WITH transaction_profile AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            COUNT(*) AS rows_per_transaction,
            COUNT(DISTINCT trn_sum_from_iss) AS iss_values,
            COUNT(DISTINCT trn_sum_from_red) AS red_values

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        COUNT(*) AS multirow_transactions,

        COUNT(*) FILTER (
            WHERE iss_values > 1
        ) AS transactions_with_varying_iss,

        COUNT(*) FILTER (
            WHERE red_values > 1
        ) AS transactions_with_varying_red

    FROM transaction_profile
    WHERE rows_per_transaction > 1
    """
).df()

display(line_value_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,multirow_transactions,transactions_with_varying_iss,transactions_with_varying_red
0,6977760,6906723,418340


In [58]:
iss_purchase_relation = duckdb_connection.execute(
    """
    WITH transaction_amounts AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            MAX(purchase_sum) AS purchase_sum,
            SUM(trn_sum_from_iss) AS summed_trn_sum_from_iss

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        COUNT(*) AS transactions,

        quantile_cont(
            ABS(summed_trn_sum_from_iss - purchase_sum),
            0.50
        ) AS median_absolute_difference,

        quantile_cont(
            ABS(summed_trn_sum_from_iss - purchase_sum),
            0.95
        ) AS p95_absolute_difference,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE ABS(summed_trn_sum_from_iss - purchase_sum) < 0.01
            ) / COUNT(*),
            2
        ) AS pct_near_exact_match

    FROM transaction_amounts
    """
).df()

display(iss_purchase_relation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,median_absolute_difference,p95_absolute_difference,pct_near_exact_match
0,8045229,0.6400,67.0000,19.5800


In [59]:
transaction_examples = duckdb_connection.execute(
    """
    SELECT
        client_id,
        transaction_id,
        transaction_datetime,
        store_id,
        product_id,
        product_quantity,
        purchase_sum,
        regular_points_received,
        regular_points_spent,
        trn_sum_from_iss,
        trn_sum_from_red
    FROM purchases_raw
    WHERE transaction_id IN (
        SELECT transaction_id
        FROM purchases_raw
        GROUP BY transaction_id
        HAVING COUNT(*) >= 5
        ORDER BY COUNT(*) DESC
        LIMIT 3
    )
    ORDER BY transaction_id, product_id
    """
).df()

styled = style_table(
    transaction_examples,
    formats={
        "product_quantity": "{:,.0f}",
        "purchase_sum": "{:,.4f}",
        "regular_points_received": "{:,.4f}",
        "regular_points_spent": "{:,.4f}",
        "trn_sum_from_iss": "{:,.4f}",
        "trn_sum_from_red": "{:,.4f}",
    },
)

transaction_start_idx = transaction_examples.index[
    ~transaction_examples["transaction_id"].duplicated()
]

styled = paint(
    styled,
    transaction_start_idx,
    [
        "client_id",
        "transaction_id",
        "transaction_datetime",
        "store_id",
        "purchase_sum",
        "regular_points_received",
        "regular_points_spent",
    ],
    "blue",
)


display(styled)


client_id,transaction_id,transaction_datetime,store_id,product_id,product_quantity,purchase_sum,regular_points_received,regular_points_spent,trn_sum_from_iss,trn_sum_from_red
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,009997f0a4,1,"8,975.0000",86.2000,0.0000,70.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,00e183631a,1,"8,975.0000",86.2000,0.0000,21.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,00e2761029,1,"8,975.0000",86.2000,0.0000,90.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,04cc351d06,1,"8,975.0000",86.2000,0.0000,40.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,04eda5782b,1,"8,975.0000",86.2000,0.0000,26.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,066894bfb7,1,"8,975.0000",86.2000,0.0000,95.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,084c815ec5,1,"8,975.0000",86.2000,0.0000,160.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,087f9acb24,1,"8,975.0000",86.2000,0.0000,120.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,0c892d6799,1,"8,975.0000",86.2000,0.0000,60.0000,—
fb4c68916a,5570455209,2019-02-20 10:54:02,2d9f6bf725,1008d9e494,1,"8,975.0000",86.2000,0.0000,46.0000,—


In [60]:
duplicate_product_within_transaction = duckdb_connection.execute(
    """
    WITH transaction_product_counts AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,
            product_id,
            COUNT(*) AS row_count
        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,
            product_id
    )

    SELECT
        COUNT(*) FILTER (
            WHERE row_count > 1
        ) AS duplicated_transaction_product_pairs,

        COALESCE(
            SUM(row_count - 1) FILTER (
                WHERE row_count > 1
            ),
            0
        ) AS extra_duplicate_rows,

        MAX(row_count) AS max_rows_for_same_product

    FROM transaction_product_counts
    """
).df()

display(duplicate_product_within_transaction)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicated_transaction_product_pairs,extra_duplicate_rows,max_rows_for_same_product
0,0,0.0000,1


In [61]:
reused_transaction_id_check = duckdb_connection.execute(
    """
    WITH transaction_identity AS (
        SELECT DISTINCT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
        FROM purchases_raw
    ),

    transaction_id_profile AS (
        SELECT
            transaction_id,
            COUNT(*) AS transaction_instances
        FROM transaction_identity
        GROUP BY transaction_id
    )

    SELECT
        COUNT(*) AS transaction_ids,

        COUNT(*) FILTER (
            WHERE transaction_instances > 1
        ) AS reused_transaction_ids,

        SUM(
            CASE
                WHEN transaction_instances > 1
                THEN transaction_instances
                ELSE 0
            END
        ) AS transaction_instances_under_reused_ids,

        MAX(transaction_instances) AS max_instances_per_transaction_id

    FROM transaction_id_profile
    """
).df()

display(reused_transaction_id_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transaction_ids,reused_transaction_ids,transaction_instances_under_reused_ids,max_instances_per_transaction_id
0,8045201,28,56.0000,2


In [62]:
composite_transaction_consistency = duckdb_connection.execute(
    """
    WITH transaction_profile AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            COUNT(DISTINCT purchase_sum) AS purchase_sum_values,
            COUNT(DISTINCT regular_points_received) AS regular_received_values,
            COUNT(DISTINCT express_points_received) AS express_received_values,
            COUNT(DISTINCT regular_points_spent) AS regular_spent_values,
            COUNT(DISTINCT express_points_spent) AS express_spent_values

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        COUNT(*) AS transactions,

        COUNT(*) FILTER (
            WHERE purchase_sum_values > 1
        ) AS inconsistent_purchase_sum,

        COUNT(*) FILTER (
            WHERE regular_received_values > 1
        ) AS inconsistent_regular_points_received,

        COUNT(*) FILTER (
            WHERE express_received_values > 1
        ) AS inconsistent_express_points_received,

        COUNT(*) FILTER (
            WHERE regular_spent_values > 1
        ) AS inconsistent_regular_points_spent,

        COUNT(*) FILTER (
            WHERE express_spent_values > 1
        ) AS inconsistent_express_points_spent

    FROM transaction_profile
    """
).df()

display(composite_transaction_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,inconsistent_purchase_sum,inconsistent_regular_points_received,inconsistent_express_points_received,inconsistent_regular_points_spent,inconsistent_express_points_spent
0,8045229,0,0,0,0,0


### Finalizing the Grain of the `purchases` Table

The `purchases` table contains **45,786,568 purchase/product-line rows**, representing **8,045,229 actual transaction instances** when a transaction is identified by:

`transaction_id + client_id + transaction_datetime + store_id`

Although there are only **8,045,201 unique `transaction_id` values**, **28 transaction IDs are reused**, with each reused ID representing two completely different transactions. Therefore, `transaction_id` alone is not enough to uniquely identify a transaction. A combination of:

`transaction_id + client_id + transaction_datetime + store_id`

is needed to correctly identify each transaction.

Within each actual transaction, a product (`product_id`) appears at most once:

* Number of duplicated transaction-product pairs: `0`
* Maximum number of rows for the same product within a transaction: `1`

Therefore, **each row in the `purchases` table represents one unique product line within a transaction**, while `product_quantity` records the number of units purchased for that product.

In summary, the columns can be grouped into two different levels of detail:

* **Transaction level (shared across the whole transaction):** `purchase_sum`, points received, points spent
* **Product-line level (specific to each product):** `product_id`, `product_quantity`, `trn_sum_from_iss`, `trn_sum_from_red`

This distinction is important because it defines how purchase history should be aggregated later when creating customer-level features. Transaction-level values should not be summed as if they were product-line values, while product-level quantities and amounts can be aggregated across the customer's purchase history.


### Remaining Extreme Numeric Values

Now that we have a clear understanding of the structure and grain of the purchase data, the next step is to take a closer look at the remaining extreme numeric values.

The goal is to determine whether these values represent **real customer purchasing behavior**, **rare but valid transactions**, or **actual data-quality issues** that need to be handled.


In [63]:
transaction_numeric_extremes = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            MAX(purchase_sum) AS purchase_sum,
            MAX(regular_points_received) AS regular_points_received,
            MAX(express_points_received) AS express_points_received,
            MAX(regular_points_spent) AS regular_points_spent,
            MAX(express_points_spent) AS express_points_spent

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        'purchase_sum' AS column_name,
        MIN(purchase_sum) AS min_value,
        quantile_cont(purchase_sum, 0.50) AS median,
        quantile_cont(purchase_sum, 0.95) AS p95,
        quantile_cont(purchase_sum, 0.99) AS p99,
        quantile_cont(purchase_sum, 0.999) AS p999,
        MAX(purchase_sum) AS max_value,
        COUNT(*) FILTER (WHERE purchase_sum < 0) AS negative_values,
        COUNT(*) FILTER (WHERE purchase_sum IS NULL) AS missing_values
    FROM transactions

    UNION ALL

    SELECT
        'regular_points_received',
        MIN(regular_points_received),
        quantile_cont(regular_points_received, 0.50),
        quantile_cont(regular_points_received, 0.95),
        quantile_cont(regular_points_received, 0.99),
        quantile_cont(regular_points_received, 0.999),
        MAX(regular_points_received),
        COUNT(*) FILTER (WHERE regular_points_received < 0),
        COUNT(*) FILTER (WHERE regular_points_received IS NULL)
    FROM transactions

    UNION ALL

    SELECT
        'express_points_received',
        MIN(express_points_received),
        quantile_cont(express_points_received, 0.50),
        quantile_cont(express_points_received, 0.95),
        quantile_cont(express_points_received, 0.99),
        quantile_cont(express_points_received, 0.999),
        MAX(express_points_received),
        COUNT(*) FILTER (WHERE express_points_received < 0),
        COUNT(*) FILTER (WHERE express_points_received IS NULL)
    FROM transactions

    UNION ALL

    SELECT
        'regular_points_spent',
        MIN(regular_points_spent),
        quantile_cont(regular_points_spent, 0.50),
        quantile_cont(regular_points_spent, 0.95),
        quantile_cont(regular_points_spent, 0.99),
        quantile_cont(regular_points_spent, 0.999),
        MAX(regular_points_spent),
        COUNT(*) FILTER (WHERE regular_points_spent < 0),
        COUNT(*) FILTER (WHERE regular_points_spent IS NULL)
    FROM transactions

    UNION ALL

    SELECT
        'express_points_spent',
        MIN(express_points_spent),
        quantile_cont(express_points_spent, 0.50),
        quantile_cont(express_points_spent, 0.95),
        quantile_cont(express_points_spent, 0.99),
        quantile_cont(express_points_spent, 0.999),
        MAX(express_points_spent),
        COUNT(*) FILTER (WHERE express_points_spent < 0),
        COUNT(*) FILTER (WHERE express_points_spent IS NULL)
    FROM transactions
    """
).df()

styled = style_table(
    transaction_numeric_extremes,
    formats={
        "min_value": "{:,.4f}",
        "median": "{:,.4f}",
        "p95": "{:,.4f}",
        "p99": "{:,.4f}",
        "p999": "{:,.4f}",
        "max_value": "{:,.4f}",
        "negative_values": "{:,.0f}",
        "missing_values": "{:,.0f}",
    },
)

styled = paint(
    styled,
    transaction_numeric_extremes.index,
    ["p999", "max_value", "negative_values", "missing_values"],
    "blue",
)

display(styled)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,min_value,median,p95,p99,p999,max_value,negative_values,missing_values
purchase_sum,0.0000,282.0000,"1,297.0000","2,372.0700","4,471.9476","35,149.0400",0,0
regular_points_received,0.0000,1.4000,15.2000,32.5000,86.0000,"2,399.0000",0,0
express_points_received,0.0000,0.0000,0.0000,0.0000,0.0000,300.0000,0,0
regular_points_spent,"-5,066.0000",0.0000,0.0000,0.0000,0.0000,0.0000,"495,331",0
express_points_spent,-300.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"90,469",0


In [64]:
line_numeric_extremes = duckdb_connection.execute(
    """
    SELECT
        'trn_sum_from_iss' AS column_name,
        MIN(trn_sum_from_iss) AS min_value,
        quantile_cont(trn_sum_from_iss, 0.50) AS median,
        quantile_cont(trn_sum_from_iss, 0.95) AS p95,
        quantile_cont(trn_sum_from_iss, 0.99) AS p99,
        quantile_cont(trn_sum_from_iss, 0.999) AS p999,
        MAX(trn_sum_from_iss) AS max_value,
        COUNT(*) FILTER (WHERE trn_sum_from_iss < 0) AS negative_values,
        COUNT(*) FILTER (WHERE trn_sum_from_iss IS NULL) AS missing_values
    FROM purchases_raw

    UNION ALL

    SELECT
        'trn_sum_from_red',
        MIN(trn_sum_from_red),
        quantile_cont(trn_sum_from_red, 0.50),
        quantile_cont(trn_sum_from_red, 0.95),
        quantile_cont(trn_sum_from_red, 0.99),
        quantile_cont(trn_sum_from_red, 0.999),
        MAX(trn_sum_from_red),
        COUNT(*) FILTER (WHERE trn_sum_from_red < 0),
        COUNT(*) FILTER (WHERE trn_sum_from_red IS NULL)
    FROM purchases_raw
    """
).df()

styled = style_table(
    line_numeric_extremes,
    formats={
        "min_value": "{:,.4f}",
        "median": "{:,.4f}",
        "p95": "{:,.4f}",
        "p99": "{:,.4f}",
        "p999": "{:,.4f}",
        "max_value": "{:,.4f}",
        "negative_values": "{:,.0f}",
        "missing_values": "{:,.0f}",
    },
)

styled = paint(
    styled,
    line_numeric_extremes.index,
    ["p999", "max_value", "missing_values"],
    "blue",
)

display(styled)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,min_value,median,p95,p99,p999,max_value,negative_values,missing_values
trn_sum_from_iss,0.0000,51.0000,200.0000,379.0000,875.0000,"35,149.0000",0,0
trn_sum_from_red,0.0000,55.0000,210.0000,390.0000,870.0000,"8,789.0000",0,"42,743,212"


In [65]:
extreme_purchase_examples = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            MAX(purchase_sum) AS purchase_sum,
            MAX(regular_points_received) AS regular_points_received,
            MAX(express_points_received) AS express_points_received,
            MAX(regular_points_spent) AS regular_points_spent,
            MAX(express_points_spent) AS express_points_spent

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT *
    FROM transactions
    ORDER BY purchase_sum DESC
    LIMIT 20
    """
).df()

styled = style_table(
    extreme_purchase_examples,
    formats={
        "purchase_sum": "{:,.4f}",
        "regular_points_received": "{:,.4f}",
        "express_points_received": "{:,.4f}",
        "regular_points_spent": "{:,.4f}",
        "express_points_spent": "{:,.4f}",
    },
)

top_purchase_idx = extreme_purchase_examples.head(5).index

styled = paint(
    styled,
    top_purchase_idx,
    ["purchase_sum", "regular_points_received", "regular_points_spent"],
    "blue",
)

display(styled)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

transaction_id,client_id,transaction_datetime,store_id,purchase_sum,regular_points_received,express_points_received,regular_points_spent,express_points_spent
9fce80e19c,8ff76eb151,2019-01-28 09:41:07,8e4ffa23c4,"35,149.0400",351.4000,0.0000,0.0000,0.0000
6f33e00bce,94b759ee31,2019-01-30 10:09:13,c3e7fca425,"34,180.8400",341.8000,0.0000,0.0000,0.0000
95f0fca761,e737b94bf8,2019-02-08 18:06:28,31f290cb5c,"29,611.4800",296.1000,0.0000,0.0000,0.0000
b8fcfb6002,c31b68e6d4,2018-12-20 15:27:45,8b9ee00e8a,"29,357.7900",293.5000,0.0000,0.0000,0.0000
2630e923e3,e4ec77f649,2018-12-27 10:43:00,9c68a5f6c1,"26,466.6000",231.0000,0.0000,0.0000,0.0000
482a36036c,d5eda4436b,2019-01-16 07:01:52,2160fd89b8,"25,913.0000",259.1000,0.0000,0.0000,0.0000
ae938726af,21801eaf7a,2018-12-21 14:12:13,07f160772f,"24,834.0000","1,065.5000",0.0000,0.0000,0.0000
5ebb3e6359,d5eda4436b,2018-12-13 13:33:37,2160fd89b8,"23,994.0000","2,399.0000",0.0000,0.0000,0.0000
69251face1,441a0953a1,2018-12-20 16:23:23,442b1ead3b,"23,599.1300",224.5000,0.0000,0.0000,0.0000
2f39ca81b1,094070e36b,2019-02-14 18:27:43,c1af19bc16,"22,939.4400",201.4000,0.0000,0.0000,0.0000


#### Follow-up Checks for Remaining Numeric Extremes

Most transaction amounts appear structurally valid: `purchase_sum` has no negative or missing values, and the largest transactions remain internally consistent.

The remaining checks therefore focus only on unusual loyalty-point behavior:

- unusually large received-point values;
- the negative sign used for spent points;
- whether missing `trn_sum_from_red` values correspond to transactions without point redemption.

In [66]:
high_received_points = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,
            MAX(purchase_sum) AS purchase_sum,
            MAX(regular_points_received) AS regular_points_received,
            MAX(express_points_received) AS express_points_received
        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        transaction_id,
        client_id,
        transaction_datetime,
        store_id,
        purchase_sum,
        regular_points_received,
        express_points_received,
        ROUND(
            100.0 * regular_points_received / NULLIF(purchase_sum, 0),
            2
        ) AS regular_points_pct_of_purchase
    FROM transactions
    ORDER BY regular_points_received DESC
    LIMIT 20
    """
).df()

styled = style_table(
    high_received_points,
    formats={
        "purchase_sum": "{:,.4f}",
        "regular_points_received": "{:,.4f}",
        "express_points_received": "{:,.4f}",
        "regular_points_pct_of_purchase": "{:,.2f}",
    },
)

top_points_idx = high_received_points.head(5).index

styled = paint(
    styled,
    top_points_idx,
    ["purchase_sum", "regular_points_received", "regular_points_pct_of_purchase"],
    "blue",
)

display(styled)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

transaction_id,client_id,transaction_datetime,store_id,purchase_sum,regular_points_received,express_points_received,regular_points_pct_of_purchase
5ebb3e6359,d5eda4436b,2018-12-13 13:33:37,2160fd89b8,"23,994.0000","2,399.0000",0.0000,10.00
25260b5e03,bb211cc796,2018-12-29 06:41:54,19515e0a19,"14,575.0000","1,457.0000",0.0000,10.00
8d0b1d6e19,6fd9883ce0,2019-02-25 15:46:42,7128be202b,"18,613.3600","1,116.6000",0.0000,6.00
ae938726af,21801eaf7a,2018-12-21 14:12:13,07f160772f,"24,834.0000","1,065.5000",0.0000,4.29
a9c0a48d58,58aeee1499,2018-12-02 10:55:13,c8b29d90ba,"15,527.6400","1,046.0000",0.0000,6.74
a3bd866e11,d5eda4436b,2018-12-09 16:45:39,2160fd89b8,"9,597.0000",959.0000,0.0000,9.99
35ffb2f171,58aeee1499,2018-12-18 11:28:32,c8b29d90ba,"13,548.1600",949.0000,0.0000,7.00
cba75be5d7,9cc359fcd7,2019-03-14 13:24:12,4f002a1816,"11,727.7900",703.2000,0.0000,6.00
42e36b5cfb,7162398ea8,2018-12-04 13:29:49,4fd99eaf4e,"10,964.0000",649.3000,0.0000,5.92
eb325d1d5c,d5eda4436b,2018-12-09 16:23:38,2160fd89b8,"5,758.0000",575.0000,0.0000,9.99


In [67]:
spent_trn_sum_from_red_relation = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id,
            client_id,
            transaction_datetime,
            store_id,

            MAX(regular_points_spent) AS regular_points_spent,
            MAX(express_points_spent) AS express_points_spent,

            COUNT(trn_sum_from_red) AS trn_sum_from_red_rows

        FROM purchases_raw
        GROUP BY
            transaction_id,
            client_id,
            transaction_datetime,
            store_id
    )

    SELECT
        COUNT(*) AS transactions,

        COUNT(*) FILTER (
            WHERE regular_points_spent < 0
               OR express_points_spent < 0
        ) AS transactions_with_points_spent,

        COUNT(*) FILTER (
            WHERE trn_sum_from_red_rows > 0
        ) AS transactions_with_trn_sum_from_red,

        COUNT(*) FILTER (
            WHERE (regular_points_spent < 0 OR express_points_spent < 0)
              AND trn_sum_from_red_rows > 0
        ) AS points_spent_and_trn_sum_from_red,

        COUNT(*) FILTER (
            WHERE (regular_points_spent < 0 OR express_points_spent < 0)
              AND trn_sum_from_red_rows = 0
        ) AS points_spent_without_trn_sum_from_red,

        COUNT(*) FILTER (
            WHERE regular_points_spent = 0
              AND express_points_spent = 0
              AND trn_sum_from_red_rows > 0
        ) AS trn_sum_from_red_without_points_spent

    FROM transactions
    """
).df()

display(spent_trn_sum_from_red_relation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,transactions_with_points_spent,transactions_with_trn_sum_from_red,points_spent_and_trn_sum_from_red,points_spent_without_trn_sum_from_red,trn_sum_from_red_without_points_spent
0,8045229,504597,506236,504592,5,1644


In [68]:
redemption_missing_profile = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,

        COUNT(*) FILTER (
            WHERE trn_sum_from_red IS NULL
        ) AS missing_red_rows,

        COUNT(*) FILTER (
            WHERE trn_sum_from_red IS NOT NULL
        ) AS nonmissing_red_rows,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE trn_sum_from_red IS NULL
            ) / COUNT(*),
            2
        ) AS pct_missing_red,

        COUNT(*) FILTER (
            WHERE trn_sum_from_red = 0
        ) AS zero_red_rows,

        COUNT(*) FILTER (
            WHERE trn_sum_from_red > 0
        ) AS positive_red_rows

    FROM purchases_raw
    """
).df()

display(redemption_missing_profile)

,purchase_rows,missing_red_rows,nonmissing_red_rows,pct_missing_red,zero_red_rows,positive_red_rows
0,45786568,42743212,3043356,93.3500,34637,3008719


### Numeric Values Conclusion

* **`purchase_sum`**: No negative or NULL values were found. The very large values come from a small number of high-value transactions and are not considered data errors.

* **`regular_points_received`, `express_points_received`**: These fields represent the number of points added to a transaction. Large values are usually found in transactions with a large `purchase_sum`, so they are kept unchanged.

* **`regular_points_spent`, `express_points_spent`**: Negative values are used to represent points that were spent or deducted. This is a reasonable way to represent the data and is therefore kept unchanged.

* **`trn_sum_from_red`**: This is a product-line-level field related to redemption. Around **93.35%** of its values are NULL. These NULLs are not treated as missing data that needs to be filled, because they may simply indicate that the product line did not involve any redemption.

* **Special case**: There are **1,644 transactions** where `trn_sum_from_red` is present but no `points_spent` is recorded. This suggests that the two groups of fields are related, but they are not exactly equivalent. Since there is not enough evidence to determine the exact business meaning of `trn_sum_from_red`, no additional interpretation is assigned to this field at this stage.

### Final Decision

**No additional numeric cleaning will be applied to the purchase-related numeric fields investigated in this section.**

* Do not remove large values in `purchase_sum` or point-received fields.
* Do not change negative values in `points_spent`.
* Do not automatically convert NULL values in `trn_sum_from_red` to `0`.

The cleaning decisions already defined for `age` and `product_quantity` will be handled separately according to their respective rules.


### Product Table Semantics

The structural checks confirmed that each `product_id` appears exactly once in the `products` table.

The remaining analysis focuses on the relationships between product metadata fields, particularly:

- whether `level_1`–`level_4` form a consistent product hierarchy;
- how `brand_id`, `segment_id`, and `vendor_id` relate to that hierarchy;
- whether missing metadata affects the interpretation of product categories.

The goal is to understand the structure of the product metadata before deciding how these fields should be used in feature engineering.

In [69]:
product_metadata_cardinality = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS products,
        COUNT(DISTINCT brand_id) AS brands,
        COUNT(DISTINCT segment_id) AS segments,
        COUNT(DISTINCT vendor_id) AS vendors,
        COUNT(DISTINCT level_1) AS level_1_values,
        COUNT(DISTINCT level_2) AS level_2_values,
        COUNT(DISTINCT level_3) AS level_3_values,
        COUNT(DISTINCT level_4) AS level_4_values
    FROM products_raw
    """
).df()

display(product_metadata_cardinality)

,products,brands,segments,vendors,level_1_values,level_2_values,level_3_values,level_4_values
0,43038,4296,116,3193,3,42,201,790


In [70]:
product_hierarchy_consistency = duckdb_connection.execute(
    """
    SELECT
        'level_2 -> level_1' AS relationship,
        COUNT(*) AS child_values,
        COUNT(*) FILTER (WHERE parent_count > 1) AS children_with_multiple_parents
    FROM (
        SELECT
            level_2,
            COUNT(DISTINCT level_1) AS parent_count
        FROM products_raw
        WHERE level_1 IS NOT NULL
          AND level_2 IS NOT NULL
        GROUP BY level_2
    )

    UNION ALL

    SELECT
        'level_3 -> level_2',
        COUNT(*),
        COUNT(*) FILTER (WHERE parent_count > 1)
    FROM (
        SELECT
            level_3,
            COUNT(DISTINCT level_2) AS parent_count
        FROM products_raw
        WHERE level_2 IS NOT NULL
          AND level_3 IS NOT NULL
        GROUP BY level_3
    )

    UNION ALL

    SELECT
        'level_4 -> level_3',
        COUNT(*),
        COUNT(*) FILTER (WHERE parent_count > 1)
    FROM (
        SELECT
            level_4,
            COUNT(DISTINCT level_3) AS parent_count
        FROM products_raw
        WHERE level_3 IS NOT NULL
          AND level_4 IS NOT NULL
        GROUP BY level_4
    )
    """
).df()


display(product_hierarchy_consistency)


,relationship,child_values,children_with_multiple_parents
0,level_2 -> level_1,42,0
1,level_3 -> level_2,201,0
2,level_4 -> level_3,790,0


In [71]:
product_hierarchy_profile = duckdb_connection.execute(
    """
    SELECT
        'level_1 -> level_2' AS relationship,
        quantile_cont(child_count, 0.50) AS median_children,
        quantile_cont(child_count, 0.95) AS p95_children,
        MAX(child_count) AS max_children
    FROM (
        SELECT
            level_1,
            COUNT(DISTINCT level_2) AS child_count
        FROM products_raw
        WHERE level_1 IS NOT NULL
          AND level_2 IS NOT NULL
        GROUP BY level_1
    )

    UNION ALL

    SELECT
        'level_2 -> level_3',
        quantile_cont(child_count, 0.50),
        quantile_cont(child_count, 0.95),
        MAX(child_count)
    FROM (
        SELECT
            level_2,
            COUNT(DISTINCT level_3) AS child_count
        FROM products_raw
        WHERE level_2 IS NOT NULL
          AND level_3 IS NOT NULL
        GROUP BY level_2
    )

    UNION ALL

    SELECT
        'level_3 -> level_4',
        quantile_cont(child_count, 0.50),
        quantile_cont(child_count, 0.95),
        MAX(child_count)
    FROM (
        SELECT
            level_3,
            COUNT(DISTINCT level_4) AS child_count
        FROM products_raw
        WHERE level_3 IS NOT NULL
          AND level_4 IS NOT NULL
        GROUP BY level_3
    )
    """
).df()


display(product_hierarchy_profile)


,relationship,median_children,p95_children,max_children
0,level_1 -> level_2,12.0000,18.3000,19
1,level_2 -> level_3,3.0000,11.9500,26
2,level_3 -> level_4,3.0000,10.0000,19


In [72]:
product_metadata_relationships = duckdb_connection.execute(
    """
    SELECT
        'brand_id' AS metadata,
        COUNT(*) AS values,
        COUNT(*) FILTER (WHERE level_1_count > 1) AS values_across_multiple_level_1,
        MAX(level_1_count) AS max_level_1_per_value
    FROM (
        SELECT
            brand_id,
            COUNT(DISTINCT level_1) AS level_1_count
        FROM products_raw
        WHERE brand_id IS NOT NULL
          AND level_1 IS NOT NULL
        GROUP BY brand_id
    )

    UNION ALL

    SELECT
        'segment_id',
        COUNT(*),
        COUNT(*) FILTER (WHERE level_1_count > 1),
        MAX(level_1_count)
    FROM (
        SELECT
            segment_id,
            COUNT(DISTINCT level_1) AS level_1_count
        FROM products_raw
        WHERE segment_id IS NOT NULL
          AND level_1 IS NOT NULL
        GROUP BY segment_id
    )

    UNION ALL

    SELECT
        'vendor_id',
        COUNT(*),
        COUNT(*) FILTER (WHERE level_1_count > 1),
        MAX(level_1_count)
    FROM (
        SELECT
            vendor_id,
            COUNT(DISTINCT level_1) AS level_1_count
        FROM products_raw
        WHERE vendor_id IS NOT NULL
          AND level_1 IS NOT NULL
        GROUP BY vendor_id
    )
    """
).df()


display(product_metadata_relationships)


,metadata,values,values_across_multiple_level_1,max_level_1_per_value
0,brand_id,4296,238,3
1,segment_id,116,15,2
2,vendor_id,3193,221,3


### Product Metadata Semantics

The product category fields form a consistent hierarchy:

`level_1 → level_2 → level_3 → level_4`

No lower-level category is associated with multiple parents, confirming that the four fields represent increasingly detailed product categories.

The hierarchy contains 3 `level_1`, 42 `level_2`, 201 `level_3`, and 790 `level_4` categories.

`brand_id`, `segment_id`, and `vendor_id` should be treated as separate product metadata rather than additional hierarchy levels. Some brands, segments, and vendors occur across multiple `level_1` categories.

Therefore, product-category features can use `level_1`–`level_4` as hierarchical information, while brand, segment, and vendor should be handled as independent categorical attributes.

### Missing-Value Handling

Missing values are handled according to the meaning and role of each column rather than with a single global rule.

- `first_redeem_date` is missing for 35,469 customers (8.86%). A missing value may represent a customer with no recorded redemption date, so no date is imputed. The missing value is preserved and will be handled explicitly when redemption-related customer features are constructed.

- Missing `brand_id`, `segment_id`, `vendor_id`, and `level_1`–`level_4` values are treated as incomplete categorical product metadata. The products themselves remain valid records and are not removed. When these fields are used as categorical features, missing values will be represented by an explicit `UNKNOWN` category.

- `netto` is missing for only 3 products. These values are retained as missing rather than replaced with an arbitrary numeric value. Imputation will only be applied later if required by the modeling pipeline.

- `trn_sum_from_red` is missing for 93.35% of purchase rows. Because its exact business meaning has not been confirmed, missing values are retained as `NULL` and are not automatically replaced with zero.

No rows are removed solely because of these missing values. Missing values are preserved until the corresponding feature is constructed and a feature-specific handling rule is required.

## Data Cleaning

Based on the data-quality investigation, the following cleaning rules are applied before downstream EDA and feature engineering:

- Invalid `age` values outside `13–100` are converted to missing.
- `first_redeem_date` values that occur slightly before `first_issue_date` on the same calendar day are normalized to `first_issue_date`.
- The single `first_redeem_date` occurring 43 days before `first_issue_date` is converted to missing.
- `product_quantity > 1000` is converted to missing.
- Missing product categorical metadata is represented by `UNKNOWN`.

No customers, products, transactions, or purchase rows are removed.

In [73]:
# ------------------------------------------------------------------
# clients_clean
# ------------------------------------------------------------------

duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW clients_clean AS

    SELECT
        client_id,

        first_issue_date,

        CASE
            -- Already missing
            WHEN first_redeem_date IS NULL
                THEN NULL

            -- Same calendar day but redeem timestamp is slightly earlier:
            -- normalize to the issue timestamp.
            WHEN first_redeem_date < first_issue_date
             AND CAST(first_redeem_date AS DATE)
                 = CAST(first_issue_date AS DATE)
                THEN first_issue_date

            -- Redeem occurs on an earlier calendar day:
            -- inconsistent date, true value cannot be recovered.
            WHEN first_redeem_date < first_issue_date
                THEN NULL

            ELSE first_redeem_date
        END AS first_redeem_date,

        CASE
            WHEN age BETWEEN 13 AND 100
                THEN age
            ELSE NULL
        END AS age,

        gender

    FROM clients_raw
    """
)


In [74]:
# ------------------------------------------------------------------
# products_clean
# ------------------------------------------------------------------

duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW products_clean AS

    SELECT
        product_id,

        COALESCE(level_1, 'UNKNOWN') AS level_1,
        COALESCE(level_2, 'UNKNOWN') AS level_2,
        COALESCE(level_3, 'UNKNOWN') AS level_3,
        COALESCE(level_4, 'UNKNOWN') AS level_4,

        COALESCE(
            CAST(segment_id AS VARCHAR),
            'UNKNOWN'
        ) AS segment_id,

        COALESCE(brand_id, 'UNKNOWN') AS brand_id,
        COALESCE(vendor_id, 'UNKNOWN') AS vendor_id,

        netto,
        is_own_trademark,
        is_alcohol

    FROM products_raw
    """
)


In [75]:
# ------------------------------------------------------------------
# purchases_clean
# ------------------------------------------------------------------

duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW purchases_clean AS

    SELECT
        client_id,
        transaction_id,
        transaction_datetime,

        regular_points_received,
        express_points_received,
        regular_points_spent,
        express_points_spent,

        purchase_sum,
        store_id,
        product_id,

        CASE
            WHEN product_quantity > 1000
                THEN NULL
            ELSE product_quantity
        END AS product_quantity,

        trn_sum_from_iss,
        trn_sum_from_red

    FROM purchases_raw
    """
)

In [76]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW uplift_train_clean AS
    SELECT *
    FROM uplift_train_raw
    """
)

In [77]:
cleaning_validation = duckdb_connection.execute(
    """
    SELECT
        -- Row preservation
        (SELECT COUNT(*) FROM clients_raw)
            AS clients_raw_rows,

        (SELECT COUNT(*) FROM clients_clean)
            AS clients_clean_rows,

        (SELECT COUNT(*) FROM products_raw)
            AS products_raw_rows,

        (SELECT COUNT(*) FROM products_clean)
            AS products_clean_rows,

        (SELECT COUNT(*) FROM purchases_raw)
            AS purchases_raw_rows,

        (SELECT COUNT(*) FROM purchases_clean)
            AS purchases_clean_rows,

        -- Age cleaning
        (
            SELECT COUNT(*)
            FROM clients_raw
            WHERE age < 13 OR age > 100
        ) AS age_values_cleaned,

        (
            SELECT COUNT(*)
            FROM clients_clean
            WHERE age < 13 OR age > 100
        ) AS invalid_age_remaining,

        -- Date cleaning
        (
            SELECT COUNT(*)
            FROM clients_raw
            WHERE first_redeem_date < first_issue_date
              AND CAST(first_redeem_date AS DATE)
                  = CAST(first_issue_date AS DATE)
        ) AS same_day_dates_normalized,

        (
            SELECT COUNT(*)
            FROM clients_clean
            WHERE first_redeem_date < first_issue_date
        ) AS inconsistent_dates_remaining,

        -- Quantity cleaning
        (
            SELECT COUNT(*)
            FROM purchases_raw
            WHERE product_quantity > 1000
        ) AS quantity_values_cleaned,

        (
            SELECT COUNT(*)
            FROM purchases_clean
            WHERE product_quantity > 1000
        ) AS invalid_quantity_remaining
    """
).df()

display(
    style_table(
        cleaning_validation,
        formats={
            column: "{:,.0f}"
            for column in cleaning_validation.columns
        },
    )
)

clients_raw_rows,clients_clean_rows,products_raw_rows,products_clean_rows,purchases_raw_rows,purchases_clean_rows,age_values_cleaned,invalid_age_remaining,same_day_dates_normalized,inconsistent_dates_remaining,quantity_values_cleaned,invalid_quantity_remaining
"400,162","400,162","43,038","43,038","45,786,568","45,786,568","1,642",0,535,0,3,0


In [78]:
CLEAN_DIR = INTERIM_DATA_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATASETS = {
    "clients": "clients_clean",
    "products": "products_clean",
    "purchases": "purchases_clean",
    "uplift_train": "uplift_train_clean",
}

clean_export_manifest = []

for dataset_name, relation_name in CLEAN_DATASETS.items():
    output_path = CLEAN_DIR / f"{dataset_name}_clean.parquet"

    if output_path.exists():
        output_path.unlink()

    duckdb_connection.execute(
        f"""
        COPY {relation_name}
        TO '{to_sql_path(output_path)}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    rows = duckdb_connection.execute(
        f"SELECT COUNT(*) FROM {relation_name}"
    ).fetchone()[0]

    clean_export_manifest.append({
        "dataset": dataset_name,
        "relation": relation_name,
        "rows": rows,
        "output_path": output_path,
        "file_size_mb": output_path.stat().st_size / 1024**2,
    })

clean_export_manifest = pd.DataFrame(clean_export_manifest)

display(
    style_table(
        clean_export_manifest,
        formats={
            "rows": "{:,.0f}",
            "file_size_mb": "{:,.2f}",
        },
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dataset,relation,rows,output_path,file_size_mb
clients,clients_clean,"400,162",d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\clients_clean.parquet,7.22
products,products_clean,"43,038",d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\products_clean.parquet,0.63
purchases,purchases_clean,"45,786,568",d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\purchases_clean.parquet,466.55
uplift_train,uplift_train_clean,"200,039",d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\uplift_train_clean.parquet,1.13
